# PCIFAR-10 

# Quick var


## Liberaries and data settup

In [1]:
# ==========================================
# BLOCK 1: SETUP & DATA LOADING
# ==========================================
import os
import gc
import json
import time
import pickle
import math
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import mixed_precision
import scipy.linalg
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from SALib.sample import saltelli
from SALib.analyze import sobol
from datetime import datetime
import json
import numpy as np
import re
import os
from SALib.analyze import sobol


2026-03-26 22:23:34.206440: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-26 22:23:34.247450: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774524214.262958    8220 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774524214.269517    8220 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774524214.298163    8220 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
# 1. Load CIFAR-10
(x_all, y_all), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# 2. Flatten and Normalize (Standard step)
x_all = x_all.reshape(-1, 3072).astype('uint8') / 255.0
x_test = x_test.reshape(-1, 3072).astype('uint8') / 255.0



# --- NEW: Train/Val Split (90% Train, 10% Val) ---
from sklearn.model_selection import train_test_split
x_train, x_val, y_train, y_val = train_test_split(
    x_all, y_all, test_size=0.1, random_state=42, stratify=y_all
)

# 4. Reshape to [Batch, Time, Channels] for your RNN
x_train = x_train[:, :, np.newaxis]
x_val   = x_val[:, :, np.newaxis]
x_test  = x_test[:, :, np.newaxis]

# 5. Labels to int32
y_train = y_train.astype('int32')
y_val   = y_val.astype('int32')
y_test  = y_test.astype('int32')

# 6. Build datasets
train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train)).shuffle(10000).batch(256)
val_ds   = tf.data.Dataset.from_tensor_slices((x_val, y_val)).batch(256)
test_ds  = tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(256)

print(f"CIFAR Ready. Train: {x_train.shape}, Val: {x_val.shape}, Test: {x_test.shape}")

/home/casper/micromamba/envs/rs/lib/python3.11/site-packages/keras/src/datasets/cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")
I0000 00:00:1774524222.703422    8220 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2156 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


CIFAR Ready. Train: (45000, 3072, 1), Val: (5000, 3072, 1), Test: (10000, 3072, 1)


## Clean environment and drift calc


In [3]:

# Forcefully disable the strict determinism flag at the TF level
try:
    tf.config.experimental.enable_op_determinism(False)
    print("Success: Strict determinism disabled.")
except:
    print("Warning: Could not toggle flag. If Phase 1 fails, restart kernel.")
def reset_seeds():
    import os
    import gc
    import random
    import numpy as np

    # 1. Clear session
    tf.keras.backend.clear_session()
    
    # 2. Disable strict determinism to allow GPU kernels to run
    os.environ['TF_DETERMINISTIC_OPS'] = '0' 
    
    # 3. Hard-lock all seeds
    os.environ['PYTHONHASHSEED'] = str(42)
    random.seed(42)
    np.random.seed(42)
    tf.random.set_seed(42)
    tf.keras.utils.set_random_seed(42) 
    
    gc.collect()
    print("Environment Reset: Seeds locked at 42. Strict determinism disabled.")

## Parameter count

In [4]:
def print_param_report(hidden_size, input_dim=1, num_classes=10):
    """
    Calculates and prints a detailed breakdown of parameters for the 
    OscillatingResonator architecture.
    """
    # RNN Layer 1: Inputs from data
    rnn1_params = (input_dim * hidden_size) + (hidden_size * hidden_size) + hidden_size
    
    # RNN Layer 2 & 3: Inputs from previous hidden layer
    rnn_mid_params = (hidden_size * hidden_size) + (hidden_size * hidden_size) + hidden_size
    
    # Final Dense Layer
    dense_params = (hidden_size * num_classes) + num_classes
    
    total = rnn1_params + (2 * rnn_mid_params) + dense_params
    
    print(f"--- PARAMETER COUNT REPORT (Hidden: {hidden_size}) ---")
    print(f"RNN Layer 1: {rnn1_params:>8,}")
    print(f"RNN Layer 2: {rnn_mid_params:>8,}")
    print(f"RNN Layer 3: {rnn_mid_params:>8,}")
    print(f"Dense Out:   {dense_params:>8,}")
    print("-" * 35)
    print(f"TOTAL PARAMS: {total:>8,}")
    print(f"Estimated Memory: {total * 4 / 1024:.2f} KB (float32)")
    print("=" * 35 + "\n")




## Variables etc

In [5]:
DATA_PERCENT = 0.01
BATCH_SIZE = 8 * 64
EPOCHS = 1         
HIDDEN = 16 
BASE_STRENGTH = 0.22 
PERIOD = 3072/4 
LAMBDA_SLOW = 0.015  
JITTER_SCALE = 0.65 
REST_BASELINE = 1.0
LEARNING_RATE = 8e-4
H_SCALE = [0.94, 0.06]    


# --- 1.1 AUTO-GENERATED SESSION NAME ---
# Creates a name like: RES_H32_S0.40_J1.15_T123456
now = datetime.now()
readable_ts = now.strftime("%y%m%d_%H%M") 
SESSION_ID = f"CIFAR-SEQ_RES_H{HIDDEN}_S{BASE_STRENGTH}_J{JITTER_SCALE}_{readable_ts}"
print(f" SESSION INITIALIZED: {SESSION_ID}")
print_param_report(HIDDEN)

 SESSION INITIALIZED: CIFAR-SEQ_RES_H16_S0.22_J0.65_260326_2223
--- PARAMETER COUNT REPORT (Hidden: 16) ---
RNN Layer 1:      288
RNN Layer 2:      528
RNN Layer 3:      528
Dense Out:        170
-----------------------------------
TOTAL PARAMS:    1,514
Estimated Memory: 5.91 KB (float32)



##  Oscillatory global field block 

In [ ]:

# --- 2. THE JITTERED CELL ---
class JitteredFeedbackCell(tf.keras.layers.Layer):
    def __init__(self, units=16, strength=0.0, period=256.0, lambda_slow=0.05, mode="active", **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.state_size = [units, units, 1] 
        self.strength = strength 
        self.period = period
        self.lambda_slow = lambda_slow
        self.mode = mode

    def build(self, input_shape):
        self.w_in = self.add_weight(shape=(input_shape[-1], self.units), name="w_in", initializer="glorot_uniform")
        self.w_rec = self.add_weight(shape=(self.units, self.units), name="w_rec",
                                     initializer=tf.keras.initializers.Orthogonal(gain=1.0))
        self.bias = self.add_weight(shape=(self.units,), name="bias", initializer="zeros")

    def call(self, inputs, states):
        prev_h, prev_G, prev_phase = states
        half = self.units // 2
        raw_signal = tf.concat([prev_h[:, half:], prev_h[:, :half]], axis=1)
        source_signal = tf.stop_gradient(raw_signal) if self.mode == "probe" else raw_signal
        new_G = (1.0 - self.lambda_slow) * prev_G + self.lambda_slow * source_signal
        G_mean = tf.reduce_mean(new_G, axis=-1, keepdims=True)
        G_std = tf.math.reduce_std(new_G, axis=-1, keepdims=True) + 1e-6
        G_norm = (new_G - G_mean) / G_std
        new_phase = prev_phase + (2.0 * math.pi / self.period)
        oscillator = tf.math.sin(new_phase)
        
        if self.mode == "active":
            bias_signal = tf.reduce_mean(source_signal, axis=-1, keepdims=True) - 0.1
            combined_signal = oscillator + (JITTER_SCALE * bias_signal)
        else:
            combined_signal = oscillator

        current_strength = self.strength * combined_signal if self.mode != "passive" else 0.0
        field_effect = REST_BASELINE + (current_strength * tf.tanh(G_norm))
        z = (tf.matmul(inputs, self.w_in) + tf.matmul(prev_h, self.w_rec) + self.bias) * field_effect
        h = (H_SCALE[0] * prev_h) + (H_SCALE[1] * tf.nn.elu(z))
        h = tf.clip_by_value(h, -20.0, 20.0)
        return h, [h, new_G, new_phase]

class OscillatingResonator(tf.keras.Model):
    def __init__(self, hidden=16, num_classes=10, strength=0.0, mode="active"):
        super().__init__()
        self.cell_ref = JitteredFeedbackCell(hidden, strength, PERIOD, LAMBDA_SLOW, mode=mode)
        self.rnn1 = tf.keras.layers.RNN(self.cell_ref, return_sequences=True)
        self.rnn2 = tf.keras.layers.RNN(JitteredFeedbackCell(hidden, strength, PERIOD, LAMBDA_SLOW, mode=mode), return_sequences=True)
        self.rnn3 = tf.keras.layers.RNN(JitteredFeedbackCell(hidden, strength, PERIOD, LAMBDA_SLOW, mode=mode), return_sequences=True)
        self.out = tf.keras.layers.Dense(num_classes)

    def call(self, x, training=False):
        h1 = self.rnn1(x, training=training)
        h2 = self.rnn2(h1, training=training)
        h3 = self.rnn3(h2, training=training)
        return self.out(h3[:, -1, :]), h3

# --- 3. UPDATED LOGGING UTILITY ---
def print_history_summary(history, model, model_name="Model", test_acc=None):
    cell = model.rnn1.cell
    total_params = model.count_params()
    
    print(f"\n DATA LOG: {model_name}")
    print(f" CONFIG: Hidden: {cell.units} | Params: {total_params:,} | F-Wgt: {cell.strength:.4f} | "
          f"L-Slow (τ): {cell.lambda_slow:.3f} | Jitter: {JITTER_SCALE:.2f} | Period: {cell.period:.1f} | Data: {DATA_PERCENT*100:.2f}%")
    print("="*145)
    header = f"{'Epoch':<6} | {'Loss':<7} | {'Val-Acc%':<8} | {'Rank':<6} | {'Sync':<6} | {'Entrp':<6} | {'A-Corr':<7} | {'Intf':<6} | {'F-Wgt':<6}"
    print(header)
    print("-" * 145)

    for i in range(len(history['loss'])):
        m = history['hidden_metrics'][i]
        print(f"{i+1:<6} | {history['loss'][i]:<7.3f} | {history['acc'][i]*100:<8.2f} | "
              f"{m['effective_rank']:<6.2f} | {m['synchrony']:<6.3f} | {m['entropy']:<6.2f} | "
              f"{m['a_corr']:<7.3f} | {m['interference']:<6.3f} | {cell.strength:<6.3f}")
    
    print("-" * 145)
    test_str = f"{test_acc*100:.2f}%" if test_acc is not None else "N/A"
    print(f" FINAL PERFORMANCE: Validation Acc: {history['acc'][-1]*100:.2f}% | TEST ACCURACY: {test_str}")
    print("="*145 + "\n")

# --- 4. TRAINING PHASE WITH SNAPSHOT & LIVE LOGS ---
# --- UPDATED TRAINING PHASE ---
def train_phase(model, train_data, val_data, epochs=3, name="model"):
    current_lr = LEARNING_RATE
    optimizer = tf.keras.optimizers.Adam(learning_rate=current_lr)
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    history = {"loss": [], "acc": [], "hidden_metrics": []}
    best_val_acc = 0.0

    @tf.function
    def train_step(x, y, lr):
        optimizer.learning_rate.assign(lr)
        with tf.GradientTape() as tape:
            logits, _ = model(x, training=True)
            loss_v = loss_fn(y, logits)
        grads = tape.gradient(loss_v, model.trainable_variables)
        grads, _ = tf.clip_by_global_norm(grads, 1.0)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        return loss_v, logits

    print(f"\n{'Epoch':<6} | {'Loss':<7} | {'Val-Acc%':<8} | {'Rank':<6} | {'Sync':<6} | {'Entrp':<6} | {'A-Corr':<7}")
    print("-" * 75)

    for epoch in range(epochs):
        acc_metric = tf.keras.metrics.SparseCategoricalAccuracy()
        epoch_loss = []
        pbar = tqdm(train_data, desc=f"EPOCH {epoch+1}/{epochs}", leave=False)
        
        for x_b, y_b in pbar:
            loss_v, logits = train_step(x_b, y_b, current_lr)
            acc_metric.update_state(y_b, logits)
            epoch_loss.append(float(loss_v))
            pbar.set_postfix({"loss": f"{np.mean(epoch_loss):.4f}", "acc": f"{acc_metric.result():.2%}"})

        # --- VALIDATION SNAPSHOT ---
        for x_v, y_v in val_data.take(1):
            logits_v, h_seq_v = model(x_v, training=False)
            val_acc = np.mean(np.argmax(logits_v.numpy(), axis=-1) == y_v.numpy())
            
            # 1. SAVE BEST WEIGHTS
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                model.save_weights(f"best_{name}_{SESSION_ID}.weights.h5")
                status_msg = f" (New Best!)"
            else:
                status_msg = ""
            
            # Metric Calculations
            h_final = h_seq_v.numpy()[:, -1, :]
            s = scipy.linalg.svdvals(h_final) + 1e-12
            p_rank = s / (np.sum(s) + 1e-10)
            eff_rank = np.exp(-np.sum(p_rank * np.log(p_rank + 1e-10)))
            
            counts, _ = np.histogram(h_final, bins=50)
            p_ent = counts / (h_final.size + 1e-10) 
            entropy_val = -np.sum(p_ent * np.log2(p_ent + 1e-10))
            
            sync_val = (np.sum(np.abs(np.corrcoef(h_final.T + 1e-8))) - HIDDEN) / (HIDDEN**2 - HIDDEN)
            acorr_val = np.mean(np.abs(np.corrcoef(h_seq_v.numpy()[0].T + 1e-8)))

            h_final = h_seq_v.numpy()[:, -1, :]
            pop_mean = np.mean(h_final, axis=1, keepdims=True)
            correlations = [
                np.corrcoef(h_final[:, i], pop_mean[:, 0])[0, 1]
                for i in range(h_final.shape[1])
            ]
            interference = np.mean(np.abs(correlations))

            history["loss"].append(np.mean(epoch_loss))
            history["acc"].append(val_acc)
            history["hidden_metrics"].append({
                "effective_rank": eff_rank,
                "synchrony": sync_val,
                "entropy": entropy_val,
                "a_corr": acorr_val,
                "interference": interference
            })

            print(f"{epoch+1:<6} | {np.mean(epoch_loss):<7.3f} | {val_acc*100:<8.2f} | "
                  f"{eff_rank:<6.2f} | {sync_val:<6.3f} | {entropy_val:<6.2f} | {acorr_val:<7.3f}{status_msg}")
    
    # 2. SAVE LATEST WEIGHTS (Post-training)
    model.save_weights(f"latest_{name}_{SESSION_ID}.weights.h5")
    print(f"--- Finished {name}: Best and Latest weights saved. ---")
            
    return history

# --- 5. SAVE ---
# This map will hold everything: configs, histories, and final scores
master_results = {
    "config": {
        "hidden": HIDDEN,
        "base_strength": BASE_STRENGTH,
        "jitter": JITTER_SCALE,
        "period": PERIOD,
        "epochs": EPOCHS,
        "data_pct": DATA_PERCENT
    },
    "runs": {}
}

def safe_evaluate(model, dataset):
    """Evaluates accuracy in batches to prevent GPU OOM errors."""
    accs = []
    for x_batch, y_batch in dataset:
        logits, _ = model(x_batch, training=False)
        preds = np.argmax(logits.numpy(), axis=-1)
        accs.append(np.mean(preds == y_batch.numpy()))
    return np.mean(accs)


# --- 6. EXECUTION ---
def reset_env():
    tf.keras.backend.clear_session()
    random.seed(42); np.random.seed(42); tf.random.set_seed(42)
    gc.collect()

# SETUP DATA
num_train = int(len(x_train) * DATA_PERCENT)
num_val = int(len(x_val)*DATA_PERCENT)
num_test  = int(len(x_test) * DATA_PERCENT)

train_ds = tf.data.Dataset.from_tensor_slices((x_train[:num_train], y_train[:num_train])) \
    .shuffle(5000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((x_val[:1000], y_val[:1000])) \
    .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_ds_subset = tf.data.Dataset.from_tensor_slices((x_test[:num_test], y_test[:num_test])) \
    .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print(f"Test Run Ready: Samples -> Train: {num_train}, Val: {num_val}, Data %: {DATA_PERCENT*100}")

# --- RUN 1: ACTIVE ---
# --- 5. EXECUTION & UNIQUE SAVING ---

master_results = {
    "session_id": SESSION_ID,
    "config": {
        "hidden": HIDDEN, "base_strength": BASE_STRENGTH,
        "jitter": JITTER_SCALE, "period": PERIOD, "epochs": EPOCHS
    },
    "runs": {}
}

def save_metrics_to_files(history, model, model_name, test_acc):
    cell = model.rnn1.cell
    total_params = model.count_params()
    
    # 1. GENERATE THE TEXT TABLE 
    log_lines = []
    log_lines.append(f"\n DATA LOG: {model_name}")
    log_lines.append(f" CONFIG: Hidden: {cell.units} | Params: {total_params:,} | F-Wgt: {cell.strength:.4f} | "
                     f"L-Slow (τ): {cell.lambda_slow:.3f} | Jitter: {JITTER_SCALE:.2f} | Period: {cell.period:.1f} | Data: {DATA_PERCENT*100:.2f}%")
    log_lines.append("="*145)
    header = f"{'Epoch':<6} | {'Loss':<7} | {'Val-Acc%':<8} | {'Rank':<6} | {'Sync':<6} | {'Entrp':<6} | {'A-Corr':<7} | {'Intf':<6} | {'F-Wgt':<6}"
    log_lines.append(header)
    log_lines.append("-" * 145)

    for i in range(len(history['loss'])):
        m = history['hidden_metrics'][i]
        line = (f"{i+1:<6} | {history['loss'][i]:<7.3f} | {history['acc'][i]*100:<8.2f} | "
                f"{m['effective_rank']:<6.2f} | {m['synchrony']:<6.3f} | {m['entropy']:<6.2f} | "
                f"{m['a_corr']:<7.3f} | {m['interference']:<6.3f} | {cell.strength:<6.3f}")
        log_lines.append(line)
    
    log_lines.append("-" * 145)
    test_str = f"{test_acc*100:.2f}%" if test_acc is not None else "N/A"
    log_lines.append(f" FINAL PERFORMANCE: Validation Acc: {history['acc'][-1]*100:.2f}% | TEST ACCURACY: {test_str}")
    log_lines.append("="*145 + "\n")

    # Save to TXT (Appends so you don't lose previous runs)
    with open(f"log_{SESSION_ID}.txt", "a") as f:
        f.write("\n".join(log_lines))
    
    # Print to console as usual
    print("\n".join(log_lines))

    # 2. SAVE RAW DATA TO JSON
    json_data = {
        "session_id": SESSION_ID,
        "model_name": model_name,
        "config": {
            "hidden": cell.units, "strength": cell.strength, "tau": cell.lambda_slow,
            "jitter": JITTER_SCALE, "period": PERIOD, "data_pct": DATA_PERCENT
        },
        "history": history,
        "test_accuracy": float(test_acc)
    }
    
    # Simple conversion for numpy types
    def clean_types(obj):
        if isinstance(obj, dict): return {k: clean_types(v) for k, v in obj.items()}
        if isinstance(obj, list): return [clean_types(i) for i in obj]
        if isinstance(obj, (np.float32, np.float64)): return float(obj)
        if isinstance(obj, (np.int32, np.int64)): return int(obj)
        return obj

    with open(f"results_{SESSION_ID}_{model_name}.json", "w") as f:
        json.dump(clean_types(json_data), f, indent=4)



Test Run Ready: Samples -> Train: 450, Val: 50, Data %: 1.0


## Confusion matrix, salience map and symmetry calculation

In [7]:
from sklearn.metrics import confusion_matrix

def plot_resonator_confusion(model, x_test_full, y_test_full, data_percent=DATA_PERCENT, reversed_mode=False, session_name="Session"):
    # 1. Respect the data constraint
    num_test = int(len(x_test_full) * data_percent)
    x_subset = x_test_full[:num_test]
    y_true = y_test_full[:num_test]

    # 2. Handle Reverse Mode (Internal labeling only)
    title_prefix = "REVERSED" if reversed_mode else "STANDARD"
    if reversed_mode:
        x_subset = x_subset[:, ::-1, :]

    # 3. Get predictions
    print(f"--> Predicting for {title_prefix}...")
    logits, _ = model.predict(x_subset, batch_size=BATCH_SIZE, verbose=0)
    y_pred = np.argmax(logits, axis=-1)
    y_true_np = y_true.numpy() if hasattr(y_true, 'numpy') else y_true

    # 4. Build ASCII Confusion Matrix
    cm = confusion_matrix(y_true_np, y_pred, labels=range(10))
    
    cm_lines = []
    cm_lines.append(f"\n{'#'*70}")
    cm_lines.append(f" {title_prefix} EVALUATION | SESSION: {session_name}")
    cm_lines.append(f"{'#'*70}")
    cm_lines.append("Pred:    0    1    2    3    4    5    6    7    8    9")
    cm_lines.append("True |" + "----" * 12)
    
    for i, row in enumerate(cm):
        row_str = f"{i:>3}  | " + " ".join([f"{val:>4}" for val in row])
        cm_lines.append(row_str)
    
    cm_lines.append("-" * 60)
    
    # 5. Calculate Bias Score
    most_common_idx = np.argmax(np.sum(cm, axis=0))
    bias_pct = (np.sum(cm[:, most_common_idx]) / np.sum(cm)) * 100
    cm_lines.append(f"Primary Bias: image '{most_common_idx}' accounts for {bias_pct:.1f}% of all predictions.")
    cm_lines.append(f"{'#'*70}\n")
    
    ascii_matrix = "\n".join(cm_lines)

    # 6. APPEND to the main session log ONLY (stripping any stray prefixes)
    # This ensures it goes into log_RES_H16...txt even if called weirdly
    clean_session = session_name.replace("Standard_", "").replace("Reversed_", "")
    log_filename = f"log_{clean_session}.txt"
    
    with open(log_filename, "a") as f:
        f.write(ascii_matrix)
    
    # Still print to console for the live check
    print(ascii_matrix)

In [8]:
def plot_field_jitter(model, sample_batch):
    # 1. Extract hidden sequences
    logits, h_seq = model(sample_batch[:1], training=False)
    h_np = h_seq[0].numpy() # Shape: (3072, HIDDEN)
    
    # 2. Reconstruct physics components
    base_delta = (2.0 * np.pi) / PERIOD
    steps = np.arange(h_np.shape[0])
    phase = steps * base_delta
    
    # Raw Oscillator and Neuronal Bias
    ghost_field = np.sin(phase)
    activity_bias = JITTER_SCALE * np.mean(h_np, axis=-1) 
    combined_signal = ghost_field + activity_bias
    
    # Reconstruct G_norm (Simplified EMA for visualization)
    # This approximates the 'new_G' EMA in your cell logic
    ema_g = 0
    g_history = []
    for step_h in h_np:
        # Shuffling the signal as per your 'half' concat logic
        half = len(step_h) // 2
        shuffled = np.concatenate([step_h[half:], step_h[:half]])
        ema_g = (1.0 - LAMBDA_SLOW) * ema_g + LAMBDA_SLOW * shuffled
        g_history.append(ema_g)
    
    g_history = np.array(g_history)
    # Z-score normalization for the tanh gating
    g_norm = (g_history - np.mean(g_history)) / (np.std(g_history) + 1e-6)
    
    # Calculate Final Gain Modulation (Field Effect)
    # FE = 1 + (strength * combined * tanh(G_norm))
    # We take the mean across neurons for the final visualization
    field_effect = 1.0 + (BASE_STRENGTH * combined_signal[:, None] * np.tanh(g_norm))
    field_effect_mean = np.mean(field_effect, axis=-1)

    # 3. Plotting
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    
    # Plot 1: Interference Pattern
    axes[0].plot(ghost_field, label="Pure Sine (Ghost)", color='gray', alpha=0.4, linestyle='--')
    axes[0].plot(combined_signal, label="Combined (Active)", color='#3498db', linewidth=1.5)
    axes[0].set_title("Carrier vs. Signal Interference")
    axes[0].legend()

    # Plot 2: Neuronal Bias (The DC Offset)
    axes[1].axhline(0, color='black', lw=1, alpha=0.3)
    axes[1].fill_between(steps, activity_bias, color='#e74c3c', alpha=0.3, label="Neuronal Bias")
    axes[1].plot(activity_bias, color='#e74c3c', lw=2)
    axes[1].set_title(f"Hidden State Pressure (Scale: {JITTER_SCALE})")
    axes[1].legend()

    # Plot 3: Resulting Gain Modulation (The Field Effect)
    axes[2].axhline(1.0, color='black', lw=1, alpha=0.3) # 1.0 is neutral gain
    axes[2].plot(field_effect_mean, color='#2ecc71', lw=2, label="Field Effect (Gain)")
    axes[2].fill_between(steps, 1.0, field_effect_mean, color='#2ecc71', alpha=0.2)
    axes[2].set_title("Total Gain Modulation ($FE_t$)")
    axes[2].set_ylabel("Multiplicative Factor")
    axes[2].legend()

    plt.tight_layout()
    plt.show()

# Run it!
sample_x, sample_y = next(iter(test_ds_subset))

In [9]:
def visualize_CIFAR_complete(model, image, label, permutation):
    # 1. Setup Inverse Permutation
    inverse_perm = np.argsort(permutation)
    # CIFAR-10 Input: (1, 3072, 1)
    img_tensor = tf.convert_to_tensor(image[np.newaxis, ...], dtype=tf.float32)
    
    # 2. Manual hidden state extraction
    h1_seq = model.rnn1(img_tensor, training=False)
    h2_seq = model.rnn2(h1_seq, training=False)
    h3_seq = model.rnn3(h2_seq, training=False)
    
    # 3. Get Prediction and Gradients for Saliency
    with tf.GradientTape() as tape:
        tape.watch(img_tensor)        
        output = model(img_tensor, training=False)
        logits = output[0] if isinstance(output, tuple) else output
        
        # --- FIX: Calculate pred_label and confidence ---
        probs = tf.nn.softmax(logits, axis=-1)
        pred_label = np.argmax(probs.numpy(), axis=-1)[0]
        confidence = np.max(probs.numpy(), axis=-1)[0]
        
        loss = logits[0, label]

    grads = tape.gradient(loss, img_tensor) 
    
    # 4. Prepare Saliency (3072 -> 32, 32, 3 -> 32, 32)
    saliency_raw = tf.reduce_max(tf.abs(grads), axis=-1).numpy().flatten()
    
    # Reshape to 3D and then take the max across channels for 2D heatmap
    unscrambled_sal = np.max(saliency_raw[inverse_perm].reshape(32, 32, 3), axis=-1)
    scrambled_sal = np.max(saliency_raw.reshape(32, 32, 3), axis=-1)

    # 5. Prepare Images (3072 -> 32, 32, 3)
    unscrambled_img = image.flatten()[inverse_perm].reshape(32, 32, 3)
    scrambled_img = image.reshape(32, 32, 3)

    # --- PLOTTING ---
    fig = plt.figure(figsize=(22, 12))
    result_color = "green" if pred_label == label else "red"
    
    # COLUMN 1: UNSCRAMBLED (RGB Image)
    ax1 = plt.subplot2grid((3, 5), (0, 0))
    ax1.imshow(unscrambled_img) 
    ax1.set_title(f"Target: {label} | PRED: {pred_label}\n({confidence*100:.1f}% Conf)", 
                  fontsize=14, color=result_color, fontweight='bold')
    
    ax2 = plt.subplot2grid((3, 5), (1, 0))
    ax2.imshow(unscrambled_sal, cmap='hot')
    ax2.set_title("Focus (Unscrambled)")

    # COLUMN 2: SCRAMBLED (RGB Image)
    ax3 = plt.subplot2grid((3, 5), (0, 1))
    ax3.imshow(scrambled_img)
    ax3.set_title("Input (Scrambled)")
    
    ax4 = plt.subplot2grid((3, 5), (1, 1))
    ax4.imshow(scrambled_sal, cmap='hot')
    ax4.set_title("Focus (Scrambled)")

    # COLUMN 3-5: THE HIDDEN HEARTBEATS
    layers = [h1_seq, h2_seq, h3_seq]
    layer_names = ["Layer 1", "Layer 2", "Layer 3: Decision State"]
    
    for i, (h_seq, name) in enumerate(zip(layers, layer_names)):
        ax = plt.subplot2grid((3, 5), (i, 2), colspan=3)
        activity = tf.transpose(h_seq[0]).numpy()
        im = ax.imshow(activity, aspect='auto', cmap='magma', interpolation='nearest')
        ax.set_title(name)
        ax.set_ylabel("Neuron ID")
        if i == 2: ax.set_xlabel("Time (3072 Steps)")
        plt.colorbar(im, ax=ax)

    plt.tight_layout()
    plt.show()

In [10]:
from scipy.spatial.distance import euclidean
from scipy.stats import pearsonr

def correlate_jitter_fields(model, image_idx, x_data, period, strength, lambda_slow):
    # 1. Get Forward Jitter
    sample_fwd = x_data[image_idx:image_idx+1]
    _, h_fwd = model(sample_fwd, training=False)
    fe_fwd = calculate_field_effect(h_fwd[0].numpy(), period, strength, lambda_slow)

    # 2. Get Reversed Jitter
    sample_rev = sample_fwd[:, ::-1, :]
    _, h_rev = model(sample_rev, training=False)
    fe_rev = calculate_field_effect(h_rev[0].numpy(), period, strength, lambda_slow)
    
    # 3. Flip the Reversed signal back to compare 1:1 with Forward
    fe_rev_flipped = fe_rev[::-1]

    # 4. Math Metrics
    corr, _ = pearsonr(fe_fwd, fe_rev_flipped)
    dist = euclidean(fe_fwd, fe_rev_flipped)
    
    print(f"\n--- SYMMETRY ANALYSIS (Hash {image_idx}) ---")
    print(f"Correlation Score (R): {corr:.4f}  (1.0 = Perfect Symmetry)")
    print(f"Euclidean Distance:    {dist:.4f}  (0.0 = Identical Field)")
    
    return corr, dist

# Helper to match your plot_field_jitter logic
def calculate_field_effect(h_np, period, strength, lambda_slow):
    steps = np.arange(len(h_np))
    ghost = np.sin(steps * (2.0 * np.pi / period))
    bias = np.mean(h_np, axis=-1)
    combined = ghost + bias
    
    # Simple EMA to simulate G_norm
    ema_g = 0
    g_history = []
    for step_h in h_np:
        half = len(step_h) // 2
        shuffled = np.concatenate([step_h[half:], step_h[:half]])
        ema_g = (1.0 - lambda_slow) * ema_g + lambda_slow * shuffled
        g_history.append(ema_g)
    
    g_norm = (np.array(g_history) - np.mean(g_history)) / (np.std(g_history) + 1e-6)
    # Mean Field Effect across neurons
    fe = 1.0 + (strength * combined[:, None] * np.tanh(g_norm))
    return np.mean(fe, axis=-1)

from scipy.stats import pearsonr
from scipy.spatial.distance import euclidean

def analyze_symmetry(model, hash_std, hash_rev, x_data):
    # 1. Generate Fields
    def get_fe(idx, reversed=False):
        batch = x_data[idx:idx+1]
        if reversed: batch = batch[:, ::-1, :]
        _, h_seq = model(batch, training=False)
        return calculate_field_effect(h_seq[0].numpy(), PERIOD, BASE_STRENGTH, LAMBDA_SLOW)

    fe_std = get_fe(hash_std, reversed=False)
    fe_rev = get_fe(hash_rev, reversed=True)
    
    # Flip the reversed field to align 1:1 with the standard timeline
    fe_rev_aligned = fe_rev[::-1]

    # 2. Math Metrics
    corr, _ = pearsonr(fe_std, fe_rev_aligned)
    dist = euclidean(fe_std, fe_rev_aligned)

    print(f"\n{'='*40}")
    print(f" FIELD SYMMETRY: Hash {hash_std} vs Hash {hash_rev}")
    print(f"{'='*40}")
    print(f" Pearson Correlation: {corr:.4f}  (Target: 1.0)")
    print(f" Euclidean Distance:  {dist:.4f}  (Target: 0.0)")
    print(f"{'='*40}\n")
    
    # Run your existing plots
    plot_field_jitter(model, x_data[hash_std:hash_std+1])
    plot_field_jitter(model, x_data[hash_rev:hash_rev+1][:, ::-1, :])





# EXECUTE 1

## Execute

In [ ]:
# --- RUN 1: ACTIVE ---

print(f"\n[Phase 1] Training Active for {SESSION_ID}...")

model_active = OscillatingResonator(hidden=HIDDEN, strength=BASE_STRENGTH, mode="active")
_ = model_active(tf.zeros((1, 3072, 1))) 

# We pass the SESSION_ID into the name so the .h5 file is unique
hist_active = train_phase(model_active, train_ds, val_ds, epochs=EPOCHS, name=f"{SESSION_ID}_active")

test_acc_active = safe_evaluate(model_active, test_ds_subset)

# Store everything in the map using the unique weight filename
master_results["runs"]["active"] = {
    "history": hist_active,
    "test_acc": float(test_acc_active),
    "weight_path": f"best_{SESSION_ID}_active.weights.h5"
}

# --- FINAL GLOBAL SAVE ---

print(f"\n SESSION COMPLETE!")
print(f"Weights: best_{SESSION_ID}_active.weights.h5")
save_metrics_to_files(hist_active, model_active, "Active", test_acc_active)

 
# --- RUN 2: PROBE ---

reset_env()
print(f"\n[Phase 2] Training Probe for {SESSION_ID}...")

model_probe = OscillatingResonator(hidden=HIDDEN, strength=BASE_STRENGTH, mode="probe")
_ = model_probe(tf.zeros((1, 3072, 1))) 

# Train with unique ID
hist_probe = train_phase(model_probe, train_ds, val_ds, epochs=EPOCHS, name=f"{SESSION_ID}_probe")

test_acc_probe = safe_evaluate(model_probe, test_ds_subset)

# Store in master map
master_results["runs"]["probe"] = {
    "history": hist_probe,
    "test_acc": float(test_acc_probe),
    "weight_path": f"best_{SESSION_ID}_probe.weights.h5"
}

# Log to files
save_metrics_to_files(hist_probe, model_probe, "Probe", test_acc_probe)

# --- RUN 3: PASSIVE ---

reset_env()
print(f"\n[Phase 3] Training Passive for {SESSION_ID}...")

model_passive = OscillatingResonator(hidden=HIDDEN, strength=BASE_STRENGTH, mode="passive")
_ = model_passive(tf.zeros((1, 3072, 1))) 

# Train with unique ID
hist_passive = train_phase(model_passive, train_ds, val_ds, epochs=EPOCHS, name=f"{SESSION_ID}_passive")

test_acc_passive = safe_evaluate(model_passive, test_ds_subset)

# Store in master map
master_results["runs"]["passive"] = {
    "history": hist_passive,
    "test_acc": float(test_acc_passive),
    "weight_path": f"best_{SESSION_ID}_passive.weights.h5"
}

# Log to files
save_metrics_to_files(hist_passive, model_passive, "Passive", test_acc_passive)


# EVAL-1

## Evaluate Confusion matrix

In [ ]:
# 1. Configuration for this eval run
DATA_PERCENT = 0.01
BATCH_SIZE = 128 
#SESSION_ID = 'RES_H16_S0.22_J0.65_260324_1110'
# 2. Dynamic weight pathing using the SESSION_ID
# This ensures we aren't accidentally evaluating the wrong model version
# --- EVALUATION BLOCK (Mode-Agnostic Version) ---

# Set this once at the top of your cell depending on which you are testing
CURRENT_MODE = "active"  # or "probe" or "passive"

# 1. Dynamic pathing (No elif needed!)
best_weight_path = f"best_{SESSION_ID}_{CURRENT_MODE}_{SESSION_ID}.weights.h5" 

print(f"--> Target Mode: {CURRENT_MODE}")
print(f"--> Weight Path: {best_weight_path}")

# 2. Universal Instantiation
if f'model_{CURRENT_MODE}' not in locals():
    print(f"--> Creating model_{CURRENT_MODE}...")
    # This works for any mode!
    model_to_eval = OscillatingResonator(hidden=HIDDEN, strength=BASE_STRENGTH, mode=CURRENT_MODE)
    _ = model_to_eval(tf.zeros((1, 3072, 1))) 
else:
    model_to_eval = locals()[f'model_{CURRENT_MODE}']

# 3. Universal Load
if os.path.exists(best_weight_path):
    model_to_eval.load_weights(best_weight_path)
    print(f"--> [SUCCESS] Weights loaded for {CURRENT_MODE}")
else:
    print(f"--> [ERROR] Weights NOT FOUND at {best_weight_path}")


if 'model_active' not in locals():
    print(f"--> model_active not found. Re-instantiating for {SESSION_ID}...")
    model_active = OscillatingResonator(hidden=HIDDEN, strength=BASE_STRENGTH, mode="active")
    _ = model_active(tf.zeros((1, 3072, 1))) 

if os.path.exists(best_weight_path):
    print(f"--> Restoring weights from: {best_weight_path}")
    model_active.load_weights(best_weight_path)
else:
    if os.path.exists("best_active.weights.h5"):
        print("--> Using generic best_active.weights.h5")
        model_active.load_weights("best_active.weights.h5")
    else:
        print(f"--> CRITICAL: No weights found for {SESSION_ID}!")


In [ ]:

# 3. Standard Evaluation
print("\nEvaluating Standard Test Set...")
num_test = int(len(x_test) * DATA_PERCENT)
x_subset = x_test[:num_test]
y_subset = y_test[:num_test]

# Using predict() for OOM safety
logits_std, _ = model_active.predict(x_subset, batch_size=BATCH_SIZE, verbose=1)
y_pred_std = np.argmax(logits_std, axis=-1)
test_acc_active = np.mean(y_pred_std == y_subset)

# 4. Reversed Evaluation
print("\nEvaluating Reversed Test Set...")
x_rev = x_subset[:, ::-1, :]
logits_rev, _ = model_active.predict(x_rev, batch_size=BATCH_SIZE, verbose=1)
y_pred_rev = np.argmax(logits_rev, axis=-1)
rev_acc_active = np.mean(y_pred_rev == y_subset) * 100

# 5. Final Stats & Logging logic
retention_ratio = rev_acc_active / (test_acc_active * 100 + 1e-9)

# Format the summary string
summary_box = [
    "\n" + "="*60,
    f" SESSION: {SESSION_ID}_{CURRENT_MODE}",
    f" REVERSAL MEMORY CHECK (Data: {DATA_PERCENT*100:.1f}%)",
    "="*60,
    f" Standard CIFAR Acc : {test_acc_active*100:>6.2f}%",
    f" Reversed CIFAR Acc : {rev_acc_active:>6.2f}%",
    f" Retention Ratio     : {retention_ratio:>6.2f}x",
    "="*60 + "\n"
]
summary_text = "\n".join(summary_box)

# APPEND to the text file
log_filename = f"log_{SESSION_ID}.txt"
with open(log_filename, "a") as f:
    f.write(summary_text)

print(summary_text)

# 6. Confusion Matrices (Already appends to log internally)
plot_resonator_confusion(model_to_eval, x_test, y_test, 
                         data_percent=DATA_PERCENT, 
                         reversed_mode=False, 
                         session_name=f"Standard_{SESSION_ID}")

plot_resonator_confusion(model_to_eval, x_test, y_test, 
                         data_percent=DATA_PERCENT, 
                         reversed_mode=True, 
                         session_name=f"Reversed_{SESSION_ID}")



In [ ]:
import os
import tensorflow as tf
import numpy as np

# --- 1. CONFIGURATION ---
DATA_PERCENT = 0.01  
BATCH_SIZE = 128
#SESSION_ID = 'RES_H16_S0.22_J0.65_260324_1110'
MODES_TO_EVAL = ["probe", "passive", "active"]
USE_LATEST = False  

# Define the exact log filename
log_filename = f"log_{SESSION_ID}.txt"

# Helper to print to console AND append to log file simultaneously
def log_print(msg):
    print(msg)
    with open(log_filename, "a") as f:
        f.write(str(msg) + "\n")

# --- 2. EVALUATION LOOP ---
for CURRENT_MODE in MODES_TO_EVAL:
    log_print("\n" + "="*70)
    log_print(f" INITIALIZING EVALUATION: {CURRENT_MODE.upper()}")
    log_print("="*70)

    # Pathing logic for 'latest' vs 'best'
    if USE_LATEST:
        weight_path = f"latest_{SESSION_ID}_{CURRENT_MODE}_{SESSION_ID}.weights.h5"
    else:
        weight_path = f"best_{SESSION_ID}_{CURRENT_MODE}_{SESSION_ID}.weights.h5"
        #best_RES_H16_S0.22_J0.65_260223_2220_probe_RES_H16_S0.22_J0.65_260223_2220.weights
        #best_probe_RES_H16_S0.22_J0.65_260223_2220.weights.h5

    log_print(f"--> Target Mode: {CURRENT_MODE}")
    log_print(f"--> Weight Path: {weight_path}")

    # 3. Model Setup
    tf.keras.backend.clear_session()
    model_to_eval = OscillatingResonator(hidden=HIDDEN, strength=BASE_STRENGTH, mode=CURRENT_MODE)
    _ = model_to_eval(tf.zeros((1, 3072, 1))) 

    # 4. Load Weights
    if os.path.exists(weight_path):
        model_to_eval.load_weights(weight_path)
        log_print(f"--> [SUCCESS] Weights loaded for {CURRENT_MODE}")
    else:
        log_print(f"--> [ERROR] Weights NOT FOUND at {weight_path}. Skipping...")
        continue

    # 5. Standard Evaluation
    log_print(f"\nEvaluating Standard Test Set ({CURRENT_MODE})...")
    num_test = int(len(x_test) * DATA_PERCENT)
    x_subset = x_test[:num_test]
    y_subset = y_test[:num_test]

    logits_std, _ = model_to_eval.predict(x_subset, batch_size=BATCH_SIZE, verbose=1)
    y_pred_std = np.argmax(logits_std, axis=-1)
    test_acc_raw = np.mean(y_pred_std == y_subset)

    # 6. Reversed Evaluation
    log_print(f"\nEvaluating Reversed Test Set ({CURRENT_MODE})...")
    x_rev = x_subset[:, ::-1, :] 
    logits_rev, _ = model_to_eval.predict(x_rev, batch_size=BATCH_SIZE, verbose=1)
    y_pred_rev = np.argmax(logits_rev, axis=-1)
    rev_acc_pct = np.mean(y_pred_rev == y_subset) * 100

    # 7. Stats calculation
    retention_ratio = rev_acc_pct / (test_acc_raw * 100 + 1e-9)
    weight_type_label = "LATEST" if USE_LATEST else "BEST"

    summary_box = [
        "\n" + "="*60,
        f" SESSION        : {SESSION_ID}",
        f" MODE           : {CURRENT_MODE.upper()} ({weight_type_label})",
        f" EVAL DATA %    : {DATA_PERCENT*100:.1f}%",
        "="*60,
        f" Standard Acc   : {test_acc_raw*100:>6.2f}%",
        f" Reversed Acc   : {rev_acc_pct:>6.2f}%",
        f" Retention Ratio: {retention_ratio:>6.2f}x",
        "="*60 + "\n"
    ]
    
    # Log the summary box
    for line in summary_box:
        log_print(line)

    # 8. Confusion Matrices 
    # (Assuming these handle their own internal logging/plotting)
    plot_resonator_confusion(model_to_eval, x_test, y_test, 
                             data_percent=DATA_PERCENT, 
                             reversed_mode=False, 
                             session_name=f"Standard_{SESSION_ID}_{CURRENT_MODE}")

    plot_resonator_confusion(model_to_eval, x_test, y_test, 
                             data_percent=DATA_PERCENT, 
                             reversed_mode=True, 
                             session_name=f"Reversed_{SESSION_ID}_{CURRENT_MODE}")

## Salience map code

In [ ]:
# Expanded scan to find the first 5 unique indices (Hashes) for each digit
def get_sample_summary_expanded(y_data):
    summary = {}
    # Target order: 1, 2, 3, 4, 5, 6, 7, 8, 9, 0
    target_order = [1, 2, 3, 4, 5, 6, 7, 8, 9, 0]
    
    for digit in target_order:
        # Find all indices where y_test matches the digit
        # Using y_data.numpy() if it's a tensor to ensure compatibility
        y_vals = y_data.numpy() if hasattr(y_data, 'numpy') else y_data
        indices = np.where(y_vals == digit)[0]
        summary[digit] = indices[:5].tolist() # Take the first five
        
    print(f"{'='*65}")
    print(f"            EXPANDED RESONATOR HASH DIRECTORY (Top 5)")
    print(f"{'='*65}")
    print(f"Image | Hash 1 | Hash 2 | Hash 3 | Hash 4 | Hash 5")
    print(f"------|--------|--------|--------|--------|--------")
    for digit in target_order:
        h = summary[digit]
        # Padding in case some images have fewer than 5 samples in the subset
        h += ["N/A"] * (5 - len(h))
        print(f"  {digit}   | {h[0]:>6} | {h[1]:>6} | {h[2]:>6} | {h[3]:>6} | {h[4]:>6}")
    print(f"{'='*65}")

# Run this on your subset
get_sample_summary_expanded(y_subset)

In [ ]:
import os
import tensorflow as tf
import numpy as np

# --- 1. CONFIGURATION ---
DATA_PERCENT = 0.01  
BATCH_SIZE = 128
#SESSION_ID = 'RES_H16_S0.22_J0.65_260324_1110'
MODES_TO_EVAL = ["active"]
USE_LATEST = True  

# Define the exact log filename
log_filename = f"log_{SESSION_ID}.txt"

# Helper to print to console AND append to log file simultaneously
def log_print(msg):
    print(msg)
    with open(log_filename, "a") as f:
        f.write(str(msg) + "\n")

# --- 2. EVALUATION LOOP ---
for CURRENT_MODE in MODES_TO_EVAL:
    log_print("\n" + "="*70)
    log_print(f" INITIALIZING EVALUATION: {CURRENT_MODE.upper()}")
    log_print("="*70)

    # Pathing logic for 'latest' vs 'best'
    if USE_LATEST:
        weight_path = f"latest_{SESSION_ID}_{CURRENT_MODE}_{SESSION_ID}.weights.h5"
    else:
        weight_path = f"best_{SESSION_ID}_{CURRENT_MODE}_{SESSION_ID}.weights.h5"
        #best_RES_H16_S0.22_J0.65_260223_2220_probe_RES_H16_S0.22_J0.65_260223_2220.weights
        #best_probe_RES_H16_S0.22_J0.65_260223_2220.weights.h5

    log_print(f"--> Target Mode: {CURRENT_MODE}")
    log_print(f"--> Weight Path: {weight_path}")

    # 3. Model Setup
    tf.keras.backend.clear_session()
    model_to_eval = OscillatingResonator(hidden=HIDDEN, strength=BASE_STRENGTH, mode=CURRENT_MODE)
    _ = model_to_eval(tf.zeros((1, 3072, 1))) 

    # 4. Load Weights
    if os.path.exists(weight_path):
        model_to_eval.load_weights(weight_path)
        log_print(f"--> [SUCCESS] Weights loaded for {CURRENT_MODE}")
    else:
        log_print(f"--> [ERROR] Weights NOT FOUND at {weight_path}. Skipping...")
        continue


In [ ]:
# Visualize normal permuted sequence

Hash_p = 6
sample_batch = x_test[Hash_p:Hash_p+1]
# Use .item() to convert the array [6] into the integer 6
#visualize_CIFAR_complete(model_to_eval, x_test[Hash_p], y_test[Hash_p].item(), perm)

# Visualize reversed permuted sequence
Hash_rev = 6
rev_batch = x_test[Hash_rev:Hash_rev+1]
sample_batch_rev = rev_batch[:, ::-1, :]
#visualize_CIFAR_complete(model_to_eval, x_rev[Hash_rev], y_test[Hash_rev], perm[::-1])

# Analyze symetry:
analyze_symmetry(model_to_eval, Hash_p, Hash_rev, x_test)


# SOBOL

## SOBOL analysis

In [11]:
import numpy as np
import tensorflow as tf
from SALib.sample import saltelli
from SALib.analyze import sobol
import matplotlib.pyplot as plt
import gc
from datetime import datetime
import json
DATA_PERCENT = 0.1
BATCH_SIZE = 32+24
EPOCHS = 10         
HIDDEN = 64 
REST_BASELINE = 1.0
LEARNING_RATE = 8e-4
N_baseline = 8 # This is the number to pass to saltelli.sample
#BASE_STRENGTH =0.6
#LAMBDA_SLOW = 0.04
#PERIOD= 3072/4 
#JITTER_SCALE = 0.6


# --- 1.1 AUTO-GENERATED SESSION NAME ---
# Creates a name like: RES_H32_S0.40_J1.15_T123456
now = datetime.now()
readable_ts = now.strftime("%y%m%d_%H%M") 
SESSION_ID = f"CIFAR_RES_H{HIDDEN}_S{BASE_STRENGTH}_J{JITTER_SCALE}_{readable_ts}"
print(f" SESSION INITIALIZED: {SESSION_ID}")
print_param_report(HIDDEN)

sobol_problem = {
    'num_vars': 5,
    'names': ['LAMBDA_SLOW', 'H_INERTIA','BASE_STRENGTH', 'PERIOD', 'JITTER_SCALE'],
    'bounds': [
        [0.001, 0.05], 
        [0.001, 0.999],
        [0.001, 0.999],      
        [(3200/32), (3200/2)], 
        [0.1, 1.2]
    ]
}

def calc_sobol_samples(problem, N, second_order=True):
    """
    Calculates total runs for a SALib Saltelli/Sobol sequence.
    Formula: N * (2D + 2) if second_order=True
             N * (D + 2)  if second_order=False
    """
    D = problem['num_vars']
    if second_order:
        total = N * (2 * D + 2)
    else:
        total = N * (D + 2)
    return total

# Your specific setup

total_runs = calc_sobol_samples(sobol_problem, N_baseline, second_order=True)

print(f"--- SOBOL RUN ESTIMATION ---")
print(f"Variables (D): {sobol_problem['num_vars']}")
print(f"Baseline (N):  {N_baseline}")
print(f"Total Model Evaluations: {total_runs}")

# Time estimation (Optional but helpful for dissertations)
avg_time_per_run = EPOCHS*240 # seconds (Estimate based on your Epochs)
total_hours = (total_runs * avg_time_per_run) / 3600
total_days = total_hours/24
print(f"Estimated Time: {total_hours:.2f} hours")
print(f"Estimated Time: {total_days:.2f} days")


 SESSION INITIALIZED: CIFAR_RES_H64_S0.22_J0.65_260326_2223
--- PARAMETER COUNT REPORT (Hidden: 64) ---
RNN Layer 1:    4,224
RNN Layer 2:    8,256
RNN Layer 3:    8,256
Dense Out:        650
-----------------------------------
TOTAL PARAMS:   21,386
Estimated Memory: 83.54 KB (float32)

--- SOBOL RUN ESTIMATION ---
Variables (D): 5
Baseline (N):  8
Total Model Evaluations: 96
Estimated Time: 64.00 hours
Estimated Time: 2.67 days


## Oscilatory 

In [ ]:

# --- 2. THE JITTERED CELL ---
class JitteredFeedbackCell(tf.keras.layers.Layer):
    def __init__(self, units=16, strength=0.0, period=256.0, lambda_slow=0.05, mode="active", **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.state_size = [units, units, 1] 
        self.strength = strength 
        self.period = period
        self.lambda_slow = lambda_slow
        self.mode = mode

    def build(self, input_shape):
        self.w_in = self.add_weight(shape=(input_shape[-1], self.units), name="w_in", initializer="glorot_uniform")
        self.w_rec = self.add_weight(shape=(self.units, self.units), name="w_rec",
                                     initializer=tf.keras.initializers.Orthogonal(gain=1.0))
        self.bias = self.add_weight(shape=(self.units,), name="bias", initializer="zeros")

    def call(self, inputs, states):
        prev_h, prev_G, prev_phase = states
        half = self.units // 2
        raw_signal = tf.concat([prev_h[:, half:], prev_h[:, :half]], axis=1)
        source_signal = tf.stop_gradient(raw_signal) if self.mode == "probe" else raw_signal
        new_G = (1.0 - self.lambda_slow) * prev_G + self.lambda_slow * source_signal
        G_mean = tf.reduce_mean(new_G, axis=-1, keepdims=True)
        G_std = tf.math.reduce_std(new_G, axis=-1, keepdims=True) + 1e-6
        G_norm = (new_G - G_mean) / G_std
        new_phase = prev_phase + (2.0 * math.pi / self.period)
        oscillator = tf.math.sin(new_phase)
        
        if self.mode == "active":
            bias_signal = tf.reduce_mean(source_signal, axis=-1, keepdims=True) - 0.1
            combined_signal = oscillator + (JITTER_SCALE * bias_signal)
        else:
            combined_signal = oscillator

        current_strength = self.strength * combined_signal if self.mode != "passive" else 0.0
        field_effect = REST_BASELINE + (current_strength * tf.tanh(G_norm))
        z = (tf.matmul(inputs, self.w_in) + tf.matmul(prev_h, self.w_rec) + self.bias) * field_effect
        h = (H_SCALE[0] * prev_h) + (H_SCALE[1] * tf.nn.elu(z))
        h = tf.clip_by_value(h, -20.0, 20.0)
        return h, [h, new_G, new_phase]

class OscillatingResonator(tf.keras.Model):
    def __init__(self, hidden=16, num_classes=10, strength=0.0, mode="active"):
        super().__init__()
        self.cell_ref = JitteredFeedbackCell(hidden, strength, PERIOD, LAMBDA_SLOW, mode=mode)
        self.rnn1 = tf.keras.layers.RNN(self.cell_ref, return_sequences=True)
        self.rnn2 = tf.keras.layers.RNN(JitteredFeedbackCell(hidden, strength, PERIOD, LAMBDA_SLOW, mode=mode), return_sequences=True)
        self.rnn3 = tf.keras.layers.RNN(JitteredFeedbackCell(hidden, strength, PERIOD, LAMBDA_SLOW, mode=mode), return_sequences=True)
        self.out = tf.keras.layers.Dense(num_classes)

    def call(self, x, training=False):
        h1 = self.rnn1(x, training=training)
        h2 = self.rnn2(h1, training=training)
        h3 = self.rnn3(h2, training=training)
        return self.out(h3[:, -1, :]), h3

# --- 3. UPDATED LOGGING UTILITY ---
def print_history_summary(history, model, model_name="Model", test_acc=None):
    cell = model.rnn1.cell
    total_params = model.count_params()
    
    print(f"\n DATA LOG: {model_name}")
    print(f" CONFIG: Hidden: {cell.units} | Params: {total_params:,} | F-Wgt: {cell.strength:.4f} | "
          f"L-Slow (τ): {cell.lambda_slow:.3f} | Jitter: {JITTER_SCALE:.2f} | Period: {cell.period:.1f} | Data: {DATA_PERCENT*100:.2f}%")
    print("="*145)
    header = f"{'Epoch':<6} | {'Loss':<7} | {'Val-Acc%':<8} | {'Rank':<6} | {'Sync':<6} | {'Entrp':<6} | {'A-Corr':<7} | {'Intf':<6} | {'F-Wgt':<6}"
    print(header)
    print("-" * 145)

    for i in range(len(history['loss'])):
        m = history['hidden_metrics'][i]
        print(f"{i+1:<6} | {history['loss'][i]:<7.3f} | {history['acc'][i]*100:<8.2f} | "
              f"{m['effective_rank']:<6.2f} | {m['synchrony']:<6.3f} | {m['entropy']:<6.2f} | "
              f"{m['a_corr']:<7.3f} | {m['interference']:<6.3f} | {cell.strength:<6.3f}")
    
    print("-" * 145)
    test_str = f"{test_acc*100:.2f}%" if test_acc is not None else "N/A"
    print(f" FINAL PERFORMANCE: Validation Acc: {history['acc'][-1]*100:.2f}% | TEST ACCURACY: {test_str}")
    print("="*145 + "\n")

# --- 4. TRAINING PHASE WITH SNAPSHOT & LIVE LOGS ---
# --- UPDATED TRAINING PHASE ---
def train_phase(model, train_data, val_data, epochs=3, name="model"):
    current_lr = LEARNING_RATE
    optimizer = tf.keras.optimizers.Adam(learning_rate=current_lr)
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    history = {"loss": [], "acc": [], "hidden_metrics": []}
    best_val_acc = 0.0

    @tf.function
    def train_step(x, y, lr):
        optimizer.learning_rate.assign(lr)
        with tf.GradientTape() as tape:
            logits, _ = model(x, training=True)
            loss_v = loss_fn(y, logits)
        grads = tape.gradient(loss_v, model.trainable_variables)
        grads, _ = tf.clip_by_global_norm(grads, 1.0)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        return loss_v, logits

    print(f"\n{'Epoch':<6} | {'Loss':<7} | {'Val-Acc%':<8} | {'Rank':<6} | {'Sync':<6} | {'Entrp':<6} | {'A-Corr':<7}")
    print("-" * 75)

    for epoch in range(epochs):
        acc_metric = tf.keras.metrics.SparseCategoricalAccuracy()
        epoch_loss = []
        pbar = tqdm(train_data, desc=f"EPOCH {epoch+1}/{epochs}", leave=False)
        
        for x_b, y_b in pbar:
            loss_v, logits = train_step(x_b, y_b, current_lr)
            acc_metric.update_state(y_b, logits)
            epoch_loss.append(float(loss_v))
            pbar.set_postfix({"loss": f"{np.mean(epoch_loss):.4f}", "acc": f"{acc_metric.result():.2%}"})

        # --- VALIDATION SNAPSHOT ---
        for x_v, y_v in val_data.take(1):
            logits_v, h_seq_v = model(x_v, training=False)
            val_acc = np.mean(np.argmax(logits_v.numpy(), axis=-1) == y_v.numpy())
            
            # 1. SAVE BEST WEIGHTS
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                model.save_weights(f"best_{name}_{SESSION_ID}.weights.h5")
                status_msg = f" (New Best!)"
            else:
                status_msg = ""
            
            # Metric Calculations
            h_final = h_seq_v.numpy()[:, -1, :]
            s = scipy.linalg.svdvals(h_final) + 1e-12
            p_rank = s / (np.sum(s) + 1e-10)
            eff_rank = np.exp(-np.sum(p_rank * np.log(p_rank + 1e-10)))
            
            counts, _ = np.histogram(h_final, bins=50)
            p_ent = counts / (h_final.size + 1e-10) 
            entropy_val = -np.sum(p_ent * np.log2(p_ent + 1e-10))
            
            sync_val = (np.sum(np.abs(np.corrcoef(h_final.T + 1e-8))) - HIDDEN) / (HIDDEN**2 - HIDDEN)
            acorr_val = np.mean(np.abs(np.corrcoef(h_seq_v.numpy()[0].T + 1e-8)))


            h_final = h_seq_v.numpy()[:, -1, :]
            pop_mean = np.mean(h_final, axis=1, keepdims=True)
            correlations = [
                np.corrcoef(h_final[:, i], pop_mean[:, 0])[0, 1]
                for i in range(h_final.shape[1])
            ]
            interference = np.mean(np.abs(correlations))

            history["loss"].append(np.mean(epoch_loss))
            history["acc"].append(val_acc)
            history["hidden_metrics"].append({
                "effective_rank": eff_rank,
                "synchrony": sync_val,
                "entropy": entropy_val,
                "a_corr": acorr_val,
                "interference": interference
            })

            print(f"{epoch+1:<6} | {np.mean(epoch_loss):<7.3f} | {val_acc*100:<8.2f} | "
                  f"{eff_rank:<6.2f} | {sync_val:<6.3f} | {entropy_val:<6.2f} | {acorr_val:<7.3f}{status_msg}")
    
    # 2. SAVE LATEST WEIGHTS (Post-training)
    model.save_weights(f"latest_{name}_{SESSION_ID}.weights.h5")
    print(f"--- Finished {name}: Best and Latest weights saved. ---")
            
    return history

# --- 5. SAVE ---
# This map will hold everything: configs, histories, and final scores
master_results = {
    "config": {
        "hidden": HIDDEN,
        "base_strength": BASE_STRENGTH,
        "jitter": JITTER_SCALE,
        "period": PERIOD,
        "epochs": EPOCHS,
        "data_pct": DATA_PERCENT
    },
    "runs": {}
}

def safe_evaluate(model, dataset):
    """Evaluates accuracy in batches to prevent GPU OOM errors."""
    accs = []
    for x_batch, y_batch in dataset:
        logits, _ = model(x_batch, training=False)
        preds = np.argmax(logits.numpy(), axis=-1)
        accs.append(np.mean(preds == y_batch.numpy()))
    return np.mean(accs)


# --- 6. EXECUTION ---
def reset_env():
    tf.keras.backend.clear_session()
    random.seed(42); np.random.seed(42); tf.random.set_seed(42)
    gc.collect()

# SETUP DATA
num_train = int(len(x_train) * DATA_PERCENT)
num_val = int(len(x_val)*DATA_PERCENT)
num_test  = int(len(x_test) * DATA_PERCENT)

train_ds = tf.data.Dataset.from_tensor_slices((x_train[:num_train], y_train[:num_train])) \
    .shuffle(5000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((x_val[:1000], y_val[:1000])) \
    .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_ds_subset = tf.data.Dataset.from_tensor_slices((x_test[:num_test], y_test[:num_test])) \
    .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print(f"Test Run Ready: Samples -> Train: {num_train}, Val: {num_val}, Data %: {DATA_PERCENT*100}")

# --- RUN 1: ACTIVE ---
# --- 5. EXECUTION & UNIQUE SAVING ---

master_results = {
    "session_id": SESSION_ID,
    "config": {
        "hidden": HIDDEN, "base_strength": BASE_STRENGTH,
        "jitter": JITTER_SCALE, "period": PERIOD, "epochs": EPOCHS
    },
    "runs": {}
}

def save_metrics_to_files(history, model, model_name, test_acc):
    cell = model.rnn1.cell
    total_params = model.count_params()
    
    # 1. GENERATE THE TEXT TABLE 
    log_lines = []
    log_lines.append(f"\n DATA LOG: {model_name}")
    log_lines.append(f" CONFIG: Hidden: {cell.units} | Params: {total_params:,} | F-Wgt: {cell.strength:.4f} | "
                     f"L-Slow (τ): {cell.lambda_slow:.3f} | Jitter: {JITTER_SCALE:.2f} | Period: {cell.period:.1f} | Data: {DATA_PERCENT*100:.2f}%")
    log_lines.append("="*145)
    header = f"{'Epoch':<6} | {'Loss':<7} | {'Val-Acc%':<8} | {'Rank':<6} | {'Sync':<6} | {'Entrp':<6} | {'A-Corr':<7} | {'Intf':<6} | {'F-Wgt':<6}"
    log_lines.append(header)
    log_lines.append("-" * 145)

    for i in range(len(history['loss'])):
        m = history['hidden_metrics'][i]
        line = (f"{i+1:<6} | {history['loss'][i]:<7.3f} | {history['acc'][i]*100:<8.2f} | "
                f"{m['effective_rank']:<6.2f} | {m['synchrony']:<6.3f} | {m['entropy']:<6.2f} | "
                f"{m['a_corr']:<7.3f} | {m['interference']:<6.3f} | {cell.strength:<6.3f}")
        log_lines.append(line)
    
    log_lines.append("-" * 145)
    test_str = f"{test_acc*100:.2f}%" if test_acc is not None else "N/A"
    log_lines.append(f" FINAL PERFORMANCE: Validation Acc: {history['acc'][-1]*100:.2f}% | TEST ACCURACY: {test_str}")
    log_lines.append("="*145 + "\n")

    # Save to TXT (Appends so you don't lose previous runs)
    with open(f"log_{SESSION_ID}.txt", "a") as f:
        f.write("\n".join(log_lines))
    
    # Print to console as usual
    print("\n".join(log_lines))

    # 2. SAVE RAW DATA TO JSON
    json_data = {
        "session_id": SESSION_ID,
        "model_name": model_name,
        "config": {
            "hidden": cell.units, "strength": cell.strength, "tau": cell.lambda_slow,
            "jitter": JITTER_SCALE, "period": PERIOD, "data_pct": DATA_PERCENT
        },
        "history": history,
        "test_accuracy": float(test_acc)
    }
    
    # Simple conversion for numpy types
    def clean_types(obj):
        if isinstance(obj, dict): return {k: clean_types(v) for k, v in obj.items()}
        if isinstance(obj, list): return [clean_types(i) for i in obj]
        if isinstance(obj, (np.float32, np.float64)): return float(obj)
        if isinstance(obj, (np.int32, np.int64)): return int(obj)
        return obj

    with open(f"results_{SESSION_ID}_{model_name}.json", "w") as f:
        json.dump(clean_types(json_data), f, indent=4)



Test Run Ready: Samples -> Train: 4500, Val: 500, Data %: 10.0


In [13]:
import tensorflow as tf
import numpy as np
import scipy.linalg
import json
import gc
import math
from tqdm.auto import tqdm

# --- 1. GLOBAL MASTER LOGGING ---
SOBOL_MASTER_DATA = {"session_id": SESSION_ID, "runs": {}, "Si_all": {}}

def update_json_log(history, model, run_id):
    """Saves every epoch's progress into one master Big Ass JSON file."""
    global SOBOL_MASTER_DATA
    if run_id is None: return # Skip if not a Sobol run
    
    cell = model.rnn1.cell
    SOBOL_MASTER_DATA["runs"][str(run_id)] = {
        "config": {
            "tau": float(cell.lambda_slow),
            "strength": float(cell.strength),
            "jitter": float(JITTER_SCALE),
            "period": float(cell.period),
            #"learning rate": float(LEARNING_RATE),
            "h_inertia": float(H_SCALE[0])
        },
        "history": history
    }

    def clean(obj):
        if isinstance(obj, dict): return {k: clean(v) for k, v in obj.items()}
        if isinstance(obj, list): return [clean(i) for i in obj]
        if isinstance(obj, (np.float32, np.float64, np.ndarray)): 
            return float(obj) if np.isscalar(obj) else obj.tolist()
        return obj

    with open(f"SOBOL_MASTER_{SESSION_ID}.json", "w") as f:
        json.dump(clean(SOBOL_MASTER_DATA), f, indent=4)

# --- 2. THE JITTERED CELL & MODEL ---
class JitteredFeedbackCell(tf.keras.layers.Layer):
    def __init__(self, units=16, strength=0.0, period=256.0, lambda_slow=0.05, mode="active", **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.state_size = [units, units, 1] 
        self.strength = strength 
        self.period = period
        self.lambda_slow = lambda_slow
        self.mode = mode

    def build(self, input_shape):
        self.w_in = self.add_weight(shape=(input_shape[-1], self.units), name="w_in", initializer="glorot_uniform")
        self.w_rec = self.add_weight(shape=(self.units, self.units), name="w_rec",
                                     initializer=tf.keras.initializers.Orthogonal(gain=1.0))
        self.bias = self.add_weight(shape=(self.units,), name="bias", initializer="zeros")

    def call(self, inputs, states):
        prev_h, prev_G, prev_phase = states
        half = self.units // 2
        raw_signal = tf.concat([prev_h[:, half:], prev_h[:, :half]], axis=1)
        source_signal = tf.stop_gradient(raw_signal) if self.mode == "probe" else raw_signal
        
        new_G = (1.0 - self.lambda_slow) * prev_G + self.lambda_slow * source_signal
        G_norm = (new_G - tf.reduce_mean(new_G, axis=-1, keepdims=True)) / (tf.math.reduce_std(new_G, axis=-1, keepdims=True) + 1e-6)
        
        new_phase = prev_phase + (2.0 * math.pi / self.period)
        oscillator = tf.math.sin(new_phase)
        
        if self.mode == "active":
            bias_signal = tf.reduce_mean(source_signal, axis=-1, keepdims=True) - 0.1
            combined_signal = oscillator + (JITTER_SCALE * bias_signal)
        else:
            combined_signal = oscillator

        current_strength = self.strength * combined_signal if self.mode != "passive" else 0.0
        field_effect = REST_BASELINE + (current_strength * tf.tanh(G_norm))
        
        z = (tf.matmul(inputs, self.w_in) + tf.matmul(prev_h, self.w_rec) + self.bias) * field_effect
        h = (H_SCALE[0] * prev_h) + (H_SCALE[1] * tf.nn.elu(z))
        h = tf.clip_by_value(h, -20.0, 20.0)
        return h, [h, new_G, new_phase]

class OscillatingResonator(tf.keras.Model):
    def __init__(self, hidden=16, num_classes=10, strength=0.0, mode="active"):
        super().__init__()
        # Use cell_ref so we can access hyperparameters easily later
        self.cell_ref = JitteredFeedbackCell(hidden, strength, PERIOD, LAMBDA_SLOW, mode=mode)
        self.rnn1 = tf.keras.layers.RNN(self.cell_ref, return_sequences=True)
        self.rnn2 = tf.keras.layers.RNN(JitteredFeedbackCell(hidden, strength, PERIOD, LAMBDA_SLOW, mode=mode), return_sequences=True)
        self.rnn3 = tf.keras.layers.RNN(JitteredFeedbackCell(hidden, strength, PERIOD, LAMBDA_SLOW, mode=mode), return_sequences=True)
        self.out = tf.keras.layers.Dense(num_classes)

    def call(self, x, training=False):
        h1 = self.rnn1(x, training=training)
        h2 = self.rnn2(h1, training=training)
        h3 = self.rnn3(h2, training=training)
        return self.out(h3[:, -1, :]), h3

# --- 3. UPDATED TRAINING PHASE ---
def train_phase(model, train_data, val_data, epochs=3, name="model", run_id=None):
    optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    history = {"loss": [], "acc": [], "hidden_metrics": []}
    best_val_acc = 0.0

    @tf.function
    def train_step(x, y):
        with tf.GradientTape() as tape:
            logits, _ = model(x, training=True)
            loss_v = loss_fn(y, logits)
        grads = tape.gradient(loss_v, model.trainable_variables)
        grads, _ = tf.clip_by_global_norm(grads, 1.0)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        return loss_v, logits

    # Added 'Intf' to the console header
    print(f"\n{'Epoch':<6} | {'Loss':<7} | {'Val-Acc%':<8} | {'Rank':<6} | {'Sync':<6} | {'Entrp':<6} | {'A-Corr':<7} | {'Intf':<6}")
    print("-" * 85)

    for epoch in range(epochs):
        epoch_losses = []
        acc_metric = tf.keras.metrics.SparseCategoricalAccuracy()
        pbar = tqdm(train_data, desc=f"EPOCH {epoch+1}/{epochs}", leave=False)
        
        for x_b, y_b in pbar:
            loss_v, logits = train_step(x_b, y_b)
            acc_metric.update_state(y_b, logits)
            epoch_losses.append(float(loss_v))
            pbar.set_postfix({"loss": f"{np.mean(epoch_losses):.4f}", "acc": f"{acc_metric.result():.2%}"})

        # --- VALIDATION SNAPSHOT ---
        for x_v, y_v in val_data.take(1):
            logits_v, h_seq_v = model(x_v, training=False)
            val_acc = np.mean(np.argmax(logits_v.numpy(), axis=-1) == y_v.numpy())            
            
            # --- METRIC CALCULATIONS ---
            h_final = h_seq_v.numpy()[:, -1, :]
            
            # 1. Effective Rank
            s = scipy.linalg.svdvals(h_final) + 1e-12
            p_rank = s / (np.sum(s) + 1e-10)
            eff_rank = np.exp(-np.sum(p_rank * np.log(p_rank + 1e-10)))
            
            # 2. Entropy
            counts, _ = np.histogram(h_final, bins=50)
            p_ent = counts / (h_final.size + 1e-10) 
            entropy_val = -np.sum(p_ent * np.log2(p_ent + 1e-10))
            
            # 3. Synchrony (Inter-neuron correlation)
            sync_val = (np.sum(np.abs(np.corrcoef(h_final.T + 1e-8))) - HIDDEN) / (HIDDEN**2 - HIDDEN)
            
            # 4. Auto-Correlation (Temporal slowness)
            acorr_val = np.mean(np.abs(np.corrcoef(h_seq_v.numpy()[0].T + 1e-8)))

            # 5. NEW: Interference (Neuron-to-Field alignment)
            # Measures how much neurons are driven by the common field vs unique input
            mean_field = np.mean(h_final, axis=1, keepdims=True)
            neuron_to_field_corrs = [
                np.abs(np.corrcoef(h_final[:, j], mean_field[:, 0])[0, 1]) 
                for j in range(h_final.shape[1])
            ]
            interference_val = np.mean(np.nan_to_num(neuron_to_field_corrs))

        # --- HISTORY UPDATE ---
        avg_loss = np.mean(epoch_losses)
        history["loss"].append(float(avg_loss))
        history["acc"].append(float(val_acc))
        history["hidden_metrics"].append({
            "effective_rank": float(eff_rank),
            "synchrony": float(sync_val),
            "entropy": float(entropy_val),
            "a_corr": float(acorr_val),
            "interference": float(interference_val)
        })

        # --- THE BIG JSON SAVE ---
        update_json_log(history, model, run_id)
        
        # Unified Console Output
        print(f"EP {epoch+1}/{epochs}: {avg_loss:<7.3f} | {val_acc:<8.2%} | {eff_rank:<6.2f} | "
              f"{sync_val:<6.3f} | {entropy_val:<6.2f} | {acorr_val:<7.3f} | {interference_val:<6.3f}")

    return history

/home/casper/micromamba/envs/rs/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Execute SOBOL

In [ ]:
# --- 7. SOBOL SENSITIVITY ANALYSIS IMPLEMENTATION (FIXED) ---

SOBOL_MASTER_DATA = {
    "session_id": SESSION_ID,
    "problem": sobol_problem,
    "runs": {},
    "Si_all": {} 
}

def evaluate_sobol_run(params, run_id):
    # Map back to the globals the model uses
    global LEARNING_RATE, JITTER_SCALE, BASE_STRENGTH, PERIOD, LAMBDA_SLOW, H_INERTIA, H_SCALE
    
    current_lslow, h_inertia, current_strength, current_period, current_jitter = params
    
    LAMBDA_SLOW = current_lslow
    H_INERTIA = h_inertia          
    BASE_STRENGTH = current_strength
    PERIOD = current_period
    JITTER_SCALE = current_jitter
    
    # Critical derivation for architectural parity
    H_SCALE = [H_INERTIA, 1.0 - H_INERTIA] 

    # Clear memory from previous run to prevent VRAM accumulation
    tf.keras.backend.clear_session()
    gc.collect()
    
    # Build model (Will use the updated globals)
    # Reset seeds here for identical initialization across configurations
    tf.keras.utils.set_random_seed(42)
    model = OscillatingResonator(hidden=HIDDEN, num_classes=10, strength=BASE_STRENGTH, mode="active")
    
    # Standard Header for the Run
    print(f"\n[Sobol Run {run_id}] "
          f"H_Inertia: {H_INERTIA:.4f} | "
          f"BASE_S: {BASE_STRENGTH:.4f} | "
          f"Tau: {LAMBDA_SLOW:.4f} | "
          f"Jitter: {JITTER_SCALE:.2f}")

    # Execute training
    # Note: ensure train_phase uses print(f"\r...", end="", flush=True) for epoch updates
    history = train_phase(model, train_ds, val_ds, epochs=EPOCHS, name="sobol", run_id=run_id)
    
    # Final newline to clear the 'flush' line from train_phase
    print("") 

    # Safely extract metrics from the final epoch
    metrics = {
        "acc": float(history['acc'][-1]),
        "rank": float(history['hidden_metrics'][-1]['effective_rank']),
        "sync": float(history['hidden_metrics'][-1]['synchrony']),
        "entr": float(history['hidden_metrics'][-1]['entropy']),
        "acorr": float(history['hidden_metrics'][-1]['a_corr']),
        "Intf": float(history['hidden_metrics'][-1]['interference'])
    }
    
    # Cleanup model reference
    del model
    return metrics

def run_sobol_analysis():
    print(f"--- STARTING MULTI-METRIC SOBOL ANALYSIS ({SESSION_ID}) ---")
    
    # SALib generates N * (2D + 2) samples
    # For D=5, N=8, this results in 8 * (10 + 2) = 96 runs
    param_values = saltelli.sample(sobol_problem, N_baseline, calc_second_order=True) 
    num_runs = len(param_values)
    
    metric_keys = ["acc", "rank", "sync", "entr", "acorr", "Intf"]
    Y = {m: np.zeros(num_runs) for m in metric_keys}

    for i, params in enumerate(param_values):
        try:
            res = evaluate_sobol_run(params, run_id=i)
            for key in metric_keys: 
                Y[key][i] = res[key]
        except Exception as e:
            # Move to next line so the error doesn't overwrite the progress line
            print(f"\n!! Run {i} failed: {e}")
            # Fill with NaN so the analyzer can handle the missing data
            for key in metric_keys: Y[key][i] = np.nan 

    # Calculate Sensitivity Indices for all tracked metrics
    Si_all = {}
    for key in metric_keys:
        y_clean = Y[key]
        nan_mask = np.isnan(y_clean)
        n_failed = np.sum(nan_mask)
        
        if n_failed > 0:
            print(f"Warning: {n_failed} failed runs for metric '{key}'")
        
        # If more than 10% of runs failed, the indices might be unreliable
        if n_failed / len(y_clean) > 0.1:
            print(f"Skipping {key}: failure rate too high ({n_failed}/{len(y_clean)})")
            Si_all[key] = None
            continue
            
        # Perform Sobol Analysis
        Si = sobol.analyze(sobol_problem, y_clean, print_to_console=False)
        Si_all[key] = {
            "S1": Si['S1'].tolist(),
            "ST": Si['ST'].tolist(),
            "S2": Si['S2'].tolist() if 'S2' in Si else None,
            "n_failed": int(n_failed)
        }

    SOBOL_MASTER_DATA["Si_all"] = Si_all
    
    # JSON Serialization helper for Numpy/Tensorflow types
    def clean(obj):
        if isinstance(obj, dict): return {k: clean(v) for k, v in obj.items()}
        if isinstance(obj, list): return [clean(i) for i in obj]
        if isinstance(obj, (np.float32, np.float64, np.ndarray)): 
            return float(obj) if np.isscalar(obj) else obj.tolist()
        return obj

    output_path = f"SOBOL_MASTER_{SESSION_ID}.json"
    with open(output_path, "w") as f:
        json.dump(clean(SOBOL_MASTER_DATA), f, indent=4)
        
    print(f"\nFINISH. Full sensitivity results saved to: {output_path}")
    return Si_all

# --- EXECUTION ---
Si_all = run_sobol_analysis()

# EVAL SOBOL

## Evaluate SOBOL

In [ ]:
#FILE_NAME = "SOBOL_MASTER_RES_H32_S0.22_J0.65_260324_1453.json"
FINAL_OUTPUT = "SOBOL_RECOVERED_S2.json"

def load_with_nan_fix(filename):
    print(f"Loading and sanitizing {filename}...")
    if not os.path.exists(filename):
        raise FileNotFoundError(f"Could not find {filename}")
        
    with open(filename, 'r', encoding='utf-8') as f:
        content = f.read()

    # The Fix: Replace unquoted NaN with null so the JSON parser accepts it
    sanitized = re.sub(r':\s*NaN', ': null', content, flags=re.IGNORECASE)
    sanitized = re.sub(r':\s*nan', ': null', sanitized)
    
    try:
        return json.loads(sanitized)
    except json.JSONDecodeError:
        # If it still fails, the file might be mid-write. Seal it.
        print("JSON incomplete, applying emergency seal...")
        if not sanitized.strip().endswith('}'):
            sanitized = sanitized.strip().rstrip(',') + "}}}}"
        return json.loads(sanitized)

# --- EXECUTION ---
try:
    master_data = load_with_nan_fix(FILE_NAME)
except Exception as e:
    print(f"FATAL ERROR during load: {e}")
    exit()

runs = master_data["runs"]
problem = master_data["problem"]
param_names = problem['names']

# Sort IDs numerically to ensure Y matches the Sobol sequence
valid_ids = sorted(runs.keys(), key=int)
num_runs = len(valid_ids)

print(f"Processing {num_runs} runs...")

# Define targets
metric_map = {
    "acc": "acc",
    "rank": "effective_rank",
    "sync": "synchrony",
    "entr": "entropy",
    "acorr": "a_corr",
    "Intf": "interference"
}

Y_data = {k: np.zeros(num_runs) for k in metric_map.keys()}

for i, rid in enumerate(valid_ids):
    hist = runs[rid].get("history", {})
    
    # 1. Grab last Accuracy
    acc_list = hist.get("acc", [])
    Y_data["acc"][i] = acc_list[-1] if acc_list else 0
    
    # 2. Grab last epoch of Hidden Metrics
    metrics_list = hist.get("hidden_metrics", [])
    if metrics_list:
        last_epoch = metrics_list[-1]
        for short_key, json_key in metric_map.items():
            if short_key == "acc": continue
            
            val = last_epoch.get(json_key)
            # If val is None (from our 'null' fix) or NaN, default to 0
            if val is None or (isinstance(val, float) and np.isnan(val)):
                Y_data[short_key][i] = 0.0
            else:
                Y_data[short_key][i] = float(val)

# --- STEP 3: DYNAMIC TRIMMING & ANALYSIS ---
Si_results = {}

D = problem['num_vars']
calc_second_order = True 
# For 5 vars: (2*5 + 2) = 12 runs per block
step_size = (2 * D + 2) if calc_second_order else (D + 2)

for key in Y_data.keys():
    total_available = len(Y_data[key])
    num_complete_blocks = total_available // step_size
    valid_cutoff = num_complete_blocks * step_size
    
    if valid_cutoff == 0:
        print(f"Skipping {key}: Need at least {step_size} runs, but only have {total_available}.")
        continue

    # Trim to valid Sobol structure
    Y_trimmed = Y_data[key][:valid_cutoff]
    
    print(f"\nAnalyzing {key.upper()}: Using {valid_cutoff} runs...")

    try:
        Si = sobol.analyze(problem, Y_trimmed, calc_second_order=calc_second_order, print_to_console=False)
        
        # --- S2 Extraction & Significance Check ---
        s2_raw = Si['S2']
        significant_interactions = []
        
        # S2 is a DxD matrix where only the upper triangle is filled
        for i in range(D):
            for j in range(i + 1, D):
                val = s2_raw[i, j]
                if not np.isnan(val) and abs(val) > 0.02:  # Threshold for significance
                    significant_interactions.append(f"{param_names[i]} x {param_names[j]}: {val:.3f}")
        
        if significant_interactions:
            print(f"  -> Top Interactions: {', '.join(significant_interactions[:2])}")

        # Store results (clean NaNs for JSON safety)
        Si_results[key] = {
            "S1": np.nan_to_num(Si['S1'], nan=0.0).tolist(),
            "ST": np.nan_to_num(Si['ST'], nan=0.0).tolist(),
            "S2": np.nan_to_num(Si['S2'], nan=0.0).tolist()
        }
        
    except Exception as e:
        print(f"  -> Sobol failed for {key}: {e}")

# --- STEP 4: SAVE ---
master_data["Si_all"] = Si_results
with open(FINAL_OUTPUT, "w") as f:
    json.dump(master_data, f, indent=4)

print(f"\nSUCCESS! Recovery and Analysis complete.")
print(f"Saved results to: {FINAL_OUTPUT}")

## Eval REAL

In [ ]:
#FILE_NAME = r"/home/casper/Documents/A_Casper/Brein/CS_Ai_Wolfhampton/Dissertation/CIFAR_P_RESULTS/H64_epoch5_Sobol_N4/SOBOL_RECOVERED_S2.json"
BASE_SAVE_PATH = r"/home/casper/Documents/A_Casper/Brein/CS_Ai_Wolfhampton/Dissertation/CIFAR_P_RESULTS/H64_epoch5_Sobol_N4/"
SAVE_DIR = BASE_SAVE_PATH

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx



# Define the metrics and their display names for the loop
metrics_to_plot = [
    ("acc", "Accuracy"),
    ("rank", "Effective Rank"),
    ("sync", "Synchrony"),
    ("entr", "Entropy"),
    ("acorr", "Auto-Correlation"),
    ("Intf", "Interference") # Matched to the key in your JSON
]

# Load Data
with open(FILE_NAME, "r") as f:
    master = json.load(f)

si_all = master["Si_all"]
param_names = master["problem"]["names"]

# --- 2. GLOBAL HEATMAPS (S1 & ST) ---
# These are "All-Metric" comparisons, so we do them once outside the loop
st_df = pd.DataFrame({m: si_all[m]["ST"] for m in si_all if si_all[m]}, index=param_names).T
s1_df = pd.DataFrame({m: si_all[m]["S1"] for m in si_all if si_all[m]}, index=param_names).T

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
sns.heatmap(s1_df, annot=True, cmap="RdBu_r", ax=axes[0], cbar_kws={'label': 'Index'})
axes[0].set_title("First-Order Sensitivity ($S_1$)\n(Direct Contribution)")

sns.heatmap(st_df, annot=True, cmap="RdBu_r", ax=axes[1], cbar_kws={'label': 'Index'})
axes[1].set_title("Total-Order Sensitivity ($S_T$)\n(Direct + Interactions)")

plt.tight_layout()
plt.savefig(os.path.join(BASE_SAVE_PATH, "global_sensitivity_comparison.png"), dpi=300)
plt.close()

# --- 3. THE MAIN ANALYSIS LOOP ---
for target_key, display_name in metrics_to_plot:
    # Check if the metric actually exists in the JSON data
    if target_key not in si_all or si_all[target_key] is None:
        print(f"Skipping {target_key}: No data found.")
        continue
        
    print(f"Generating plots for: {display_name}...")
    
    # Create subfolder for this metric
    metric_path = os.path.join(BASE_SAVE_PATH, target_key)
    os.makedirs(metric_path, exist_ok=True)
    
    # A. PREPARE S2 MATRIX
    s2_matrix = pd.DataFrame(
        si_all[target_key]["S2"], 
        index=param_names, 
        columns=param_names
    ).astype(float)

    # B. PLOT: S2 HEATMAP
    plt.figure(figsize=(10, 8))
    mask = np.tril(np.ones_like(s2_matrix, dtype=bool))
    sns.heatmap(
        s2_matrix, mask=mask, annot=True, cmap="RdBu_r", 
        center=0, fmt=".3f", cbar_kws={'label': 'Synergy Index'}
    )
    plt.title(f"Second-Order Synergy ($S_2$): {display_name}\n(Red=Synergy | Blue=Competition)")
    plt.savefig(os.path.join(metric_path, f"{target_key}_s2_heatmap.png"), dpi=300)
    plt.close()

    # C. PLOT: INTERACTION NETWORK
    plt.figure(figsize=(8, 8))
    G = nx.Graph()

    for i, p1 in enumerate(param_names):
        for j, p2 in enumerate(param_names):
            if i >= j: continue 
            val = s2_matrix.iloc[i, j]
            
            # Thresholding for clarity: only plot interactions > 0.02
            if not np.isnan(val) and abs(val) > 0.02: 
                color = 'firebrick' if val > 0 else 'royalblue'
                G.add_edge(p1, p2, weight=abs(val), color=color)

    if G.number_of_edges() > 0:
        pos = nx.spring_layout(G, k=1.2, seed=42)
        edges = G.edges()
        colors = [G[u][v]['color'] for u, v in edges]
        weights = [G[u][v]['weight'] * 10 for u, v in edges] # Scale factor

        nx.draw(G, pos, with_labels=True, node_size=3500, node_color="#f8f8f8", 
                edge_color=colors, width=weights, font_size=9, font_weight="bold",
                edge_cmap=plt.cm.RdBu)
        
        plt.title(f"Parameter Interaction Network: {display_name}")
        plt.savefig(os.path.join(metric_path, f"{target_key}_network.png"), dpi=300)
    else:
        print(f"Note: No significant interactions for {display_name} to plot in network.")
    
    plt.close()

print(f"\nSUCCESS: All plots saved to {os.path.abspath(BASE_SAVE_PATH)}")

In [ ]:

# Mapping the "Problem" names to the "Config" keys found in your JSON
CONFIG_MAP = {
    "LAMBDA_SLOW": "tau",
    "BASE_STRENGTH": "strength",
    "JITTER_SCALE": "jitter",
    "PERIOD": "period",
    "H_INERTIA": "h_inertia",
    "LEARNING_RATE": "learning rate"
}

# List of all measures to plot (Key, Display Name)
METRICS_TO_PLOT = [
    ("acc", "Accuracy"),
    ("loss", "Loss"),
    ("rank", "Effective Rank"),
    ("sync", "Synchrony"),
    ("entr", "Entropy"),
    ("acorr", "Auto-Correlation"),
    ("Intf", "Interference")
]

def generate_sobol_evolution_plots(file_path):
    if not os.path.exists(file_path):
        print(f"Error: {file_path} not found.")
        return

    with open(file_path, "r") as f:
        data = json.load(f)

    problem = data["problem"]
    runs = data["runs"]
    param_names = problem["names"]
    param_bounds = problem["bounds"]

    # Iterate through every metric (e.g., Accuracy, then Rank, etc.)
    for metric_key, metric_label in METRICS_TO_PLOT:
        print(f"Generating evolution plots for: {metric_label}...")
        
        # Create subfolder for this specific metric
        metric_path = os.path.join(BASE_SAVE_PATH, metric_key)
        os.makedirs(metric_path, exist_ok=True)
        
        # Create a figure with 5 subplots (one for each hyperparameter)
        # Note: If you have 6 parameters (including LR), change 5 to 6
        num_params = len(param_names)
        fig, axes = plt.subplots(1, num_params, figsize=(5 * num_params, 7), sharey=False)
        
        fig.suptitle(f"Drift of {metric_label} Across Parameter Space (Epoch 1 → Final)", 
                     fontsize=22, fontweight='bold', y=1.05)
        
        for i, p_name in enumerate(param_names):
            # Handle cases with single vs multiple subplots
            ax = axes[i] if num_params > 1 else axes
            
            c_key = CONFIG_MAP.get(p_name)
            bounds = param_bounds[i]
            
            # Formatting the subplot
            ax.set_title(f"vs {p_name}", fontsize=14, pad=10)
            ax.set_ylabel(f"Parameter Value: {p_name}", fontsize=12)
            ax.set_xlabel(f"{metric_label} Value", fontsize=12)
            ax.set_ylim(bounds[0], bounds[1])
            ax.grid(True, linestyle="--", alpha=0.3)
            
            # Plot trajectories for every single run
            for run_id, run_data in runs.items():
                try:
                    if "history" not in run_data or "config" not in run_data:
                        continue
                    
                    # 1. Get the hyperparameter value (Constant for the run - Y-axis)
                    y_val = run_data["config"][c_key]
                    
                    # 2. Get the metric history (Changes per epoch - X-axis)
                    if metric_key in ["acc", "loss"]:
                        x_history = run_data["history"][metric_key]
                    else:
                        # Extract from the hidden_metrics list (rank, sync, etc.)
                        x_history = [epoch[metric_key] for epoch in run_data["history"]["hidden_metrics"]]
                    
                    # 3. Draw the line (Blue = trajectory, Red Dot = Final State)
                    ax.plot(x_history, [y_val] * len(x_history), 
                            color='#2980b9', alpha=0.08, linewidth=1)
                    
                    # Mark the final epoch
                    ax.scatter(x_history[-1], y_val, color='#c0392b', s=8, alpha=0.2)
                    
                except (KeyError, IndexError, TypeError):
                    continue 
        
        plt.tight_layout()
        
        # Save high-res PNG inside the specific metric folder
        save_filename = f"EVOLUTION_{metric_key.upper()}.png"
        save_fullpath = os.path.join(metric_path, save_filename)
        
        plt.savefig(save_fullpath, dpi=200, bbox_inches='tight')
        print(f"Saved: {save_fullpath}")
        
        # Close to save memory
        plt.close(fig)

if __name__ == "__main__":
    generate_sobol_evolution_plots(FILE_NAME)
    print(f"\nSUCCESS: All evolution plots saved to {BASE_SAVE_PATH}")

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pandas.plotting import parallel_coordinates


# Mapping JSON config keys to the Sobol Problem names
CONFIG_MAP = {
    "LAMBDA_SLOW": "tau", 
    "BASE_STRENGTH": "strength", 
    "JITTER_SCALE": "jitter", 
    "PERIOD": "period", 
    "H_INERTIA": "h_inertia",
    "LEARNING_RATE": "learning rate"
}

METRICS_TO_PLOT = [
    ("acc", "Accuracy"),
    ("loss", "Loss"),
    ("rank", "Effective Rank"),
    ("sync", "Synchrony"),
    ("entr", "Entropy"),
    ("acorr", "Auto-Correlation"),
    ("intf", "Interference")
]

# --- 2. DATA PROCESSING ---
def get_processed_df(data):
    runs = data["runs"]
    param_names = data["problem"]["names"]
    
    records = []
    for rid, rdata in runs.items():
        if "history" not in rdata: continue
        h = rdata["history"]
        m = h["hidden_metrics"][-1] # Take the final epoch state
        
        row = {
            "acc": h["acc"][-1],
            "loss": h["loss"][-1],
            "rank": m["effective_rank"],
            "sync": m["synchrony"],
            "entr": m["entropy"],
            "acorr": m["a_corr"],
            "intf": m["interference"]
        }
        # Map parameters from config to the Sobol problem names
        for p_name in param_names:
            c_key = CONFIG_MAP.get(p_name, p_name.lower())
            row[p_name] = rdata["config"].get(c_key, 0)
        records.append(row)
    return pd.DataFrame(records)

with open(FILE_NAME, "r") as f:
    data = json.load(f)

df = get_processed_df(data)
param_cols = data["problem"]["names"]

# --- 3. GLOBAL CORRELATION MATRIX ---
plt.figure(figsize=(12, 10))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, cmap="coolwarm", fmt=".2f", center=0)
plt.title("Global Parameter-Metric Correlation Matrix", fontsize=16, pad=20)
plt.savefig(os.path.join(BASE_SAVE_PATH, "global_metric_correlation.png"), dpi=300)
plt.close()

# --- 4. PER-METRIC PARALLEL COORDINATES LOOP ---
for target_key, display_name in METRICS_TO_PLOT:
    if target_key not in df.columns: continue
    
    print(f"Generating Parallel Coordinates for: {display_name}...")
    
    # Create subfolder
    metric_path = os.path.join(BASE_SAVE_PATH, target_key)
    os.makedirs(metric_path, exist_ok=True)

    # Prepare data for Parallel Coordinates
    # We categorize the metric into 4 bins (Quartiles) to see which paths lead to "High" values
    plot_df = df[param_cols + [target_key]].copy()
    
    # Create a categorical 'Performance' column for coloring
    # Note: For 'loss', 'Low' is actually 'Good', but we keep logic consistent for now
    plot_df['Performance'] = pd.qcut(plot_df[target_key], 4, labels=['Low', 'Mid-Low', 'Mid-High', 'High'])

    plt.figure(figsize=(15, 7))
    
    # Use a custom color palette (Red for High, Blue for Low)
    palette = {"High": "#e74c3c", "Mid-High": "#f39c12", "Mid-Low": "#3498db", "Low": "#2c3e50"}
    
    # The actual Parallel Coordinates plot
    parallel_coordinates(plot_df.drop(columns=[target_key]), 'Performance', 
                         color=[palette[k] for k in plot_df['Performance'].unique()],
                         alpha=0.4, linewidth=1.5)

    plt.title(f"High-Dimensional Parameter Paths for {display_name}\n(Colored by Performance Quartile)", 
              fontsize=18, fontweight='bold', pad=20)
    plt.ylabel("Normalized Parameter Value Range")
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.legend(title="Quartile Tier", loc='upper right', bbox_to_anchor=(1.1, 1))
    
    save_path = os.path.join(metric_path, f"PARALLEL_PATHS_{target_key.upper()}.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

print(f"\nCOMPLETED: All high-dimensional analysis saved to {BASE_SAVE_PATH}")

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


# --- 2. PLOT 1: RADAR SENSITIVITY (ST) ---
def plot_radar(si_all, param_names, target_metrics=['acc', 'sync', 'rank', 'entr', 'acorr', 'Intf']):
    print("Generating Radar Sensitivity Plot...")
    angles = np.linspace(0, 2 * np.pi, len(param_names), endpoint=False).tolist()
    angles += angles[:1] # Close the circle
    
    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
    
    for m_key in target_metrics:
        if m_key in si_all and si_all[m_key] is not None:
            values = np.array(si_all[m_key]["ST"]).tolist()
            values += values[:1]
            ax.plot(angles, values, linewidth=2.5, label=m_key.upper(), alpha=0.8)
            ax.fill(angles, values, alpha=0.05)

    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(param_names, fontsize=12)
    
    plt.legend(loc='upper right', bbox_to_anchor=(1.2, 1.1), title="Metric $S_T$")
    plt.title("Total Sensitivity ($S_T$) Landscape across All Metrics", pad=30, fontsize=18, fontweight='bold')
    
    plt.savefig(os.path.join(BASE_SAVE_PATH, "MECHANISM_Radar_Sensitivity.png"), dpi=300, bbox_inches='tight')
    plt.close()

# --- 3. PLOT 2: PHASE PORTRAIT (Synchrony vs Accuracy) ---
def plot_phase_portrait(df):
    print("Generating Phase Portrait...")
    plt.figure(figsize=(12, 8))
    
    # Using 'sync' and 'acc' from processed dataframe
    scatter = plt.scatter(df['sync'], df['acc'], c=df['H_INERTIA'], cmap='viridis', s=40, alpha=0.7, edgecolors='w', linewidth=0.5)
    
    cbar = plt.colorbar(scatter)
    cbar.set_label('H_INERTIA (Temporal Stability)', fontsize=12)
    
    plt.xlabel('Global Synchrony (Final State)', fontsize=13)
    plt.ylabel('Final Accuracy', fontsize=13)
    plt.title('Mechanism Analysis: Accuracy vs. Synchrony Phase Portrait', fontsize=16, fontweight='bold', pad=20)
    plt.grid(True, linestyle='--', alpha=0.4)
    
    plt.savefig(os.path.join(BASE_SAVE_PATH, "MECHANISM_Sync_Acc_Phase.png"), dpi=300, bbox_inches='tight')
    plt.close()

# --- 4. PLOT 3: NEURAL PATHWAY ANALYSIS (Parallel Coordinates) ---
def plot_pathway_analysis(df, target_col='acc', quantile=0.99, title_prefix="Winning Recipe"):
    print(f"Generating Pathway Analysis for {target_col}...")
    
    # 1. Prepare and Normalize Data
    # Mapping to the columns present in your processed DF
    cols = ['LAMBDA_SLOW', 'BASE_STRENGTH', 'JITTER_SCALE', 'PERIOD', 'H_INERTIA', target_col]
    df_plot = df[cols].copy()
    
    # Min-Max Normalization for visual alignment [0, 1]
    for col in cols:
        df_plot[col] = (df_plot[col] - df_plot[col].min()) / (df_plot[col].max() - df_plot[col].min())
    
    # 2. Split into groups based on Quantile
    q_val = df[target_col].quantile(quantile)
    top_indices = df[df[target_col] >= q_val].index
    other_indices = df[df[target_col] < q_val].index
    
    x = np.arange(len(cols))
    plt.figure(figsize=(16, 9))
    
    # 3. Plot Baseline (Deep Red, Low Alpha)
    for idx in other_indices:
        plt.plot(x, df_plot.loc[idx], color="#710d0d", alpha=0.1, linewidth=0.5)
        
    # 4. Plot Top Performers (Bright Green highlight)
    for idx in top_indices:
        plt.plot(x, df_plot.loc[idx], color="#3ce73c", alpha=0.7, linewidth=2.5)

    # 5. Formatting
    plt.xticks(x, cols, rotation=15, fontsize=12)
    plt.title(f"{title_prefix}: Top 1% {target_col.upper()} (Green) vs Others (Red)", fontsize=18, fontweight='bold', pad=25)
    plt.ylabel("Normalized Parameter Range [0-1]", fontsize=13)
    
    legend_elements = [
        Line2D([0], [0], color="#710d0d", lw=1, alpha=0.3, label='Baseline Runs'),
        Line2D([0], [0], color="#3ce73c", lw=3, alpha=0.8, label=f'Top 1% Performers ({target_col})')
    ]
    plt.legend(handles=legend_elements, loc='upper right')
    
    plt.grid(axis='x', linestyle='-', alpha=0.2)
    plt.grid(axis='y', linestyle='--', alpha=0.3)
    plt.tight_layout()
    
    save_name = f"PATHWAY_{target_col.upper()}_Winning_Recipe.png"
    plt.savefig(os.path.join(BASE_SAVE_PATH, save_name), dpi=300)
    plt.close()

# --- EXECUTE SUITE ---
if __name__ == "__main__":
    # Ensure df is loaded from your previous processing script
    # df = get_processed_df(data) 
    
    # 1. Sensitivity Radar
    plot_radar(data["Si_all"], data["problem"]["names"])
    
    # 2. Physical Mechanism Phase Portrait
    plot_phase_portrait(df)
    
    # 3. Winning Recipe for Accuracy
    plot_pathway_analysis(df, target_col='acc', quantile=0.99, title_prefix="Neural Pathway Analysis")
    
    # 4. Winning Recipe for Dimensionality (Rank)
    plot_pathway_analysis(df, target_col='rank', quantile=0.99, title_prefix="Dimensionality Analysis")

    print(f"\nALL MECHANISM PLOTS EXPORTED TO: {BASE_SAVE_PATH}")

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd
from itertools import combinations

# --- 1. SETTINGS ---
BIFURCATION_DIR = os.path.join(BASE_SAVE_PATH, "bifurcation_comparisons_full_range")
os.makedirs(BIFURCATION_DIR, exist_ok=True)

# Define metrics (ensure keys 'intf' and 'rank' match your DataFrame columns)
METRICS = [
    ('acc', 'Accuracy', '#2ecc71'),
    ('rank', 'Effective Rank', '#3498db'),
    ('sync', 'Synchrony', '#9b59b6'),
    ('intf', 'Interference', '#e67e22'),
    ('entr', 'Entropy', '#f1c40f'),
    ('acorr', 'Auto-Correlation', '#1abc9c')
]

def generate_all_bifurcation_pairs(df):
    print(f"Starting Full-Range Bifurcation Analysis [0.0 - 1.0]...")
    
    # Ensure data is sorted by the 'Master Parameter'
    df_sweep = df.sort_values('H_INERTIA').copy()
    
    metric_pairs = list(combinations(METRICS, 2))
    
    for (m1_key, m1_name, m1_color), (m2_key, m2_name, m2_color) in metric_pairs:
        if m1_key not in df.columns or m2_key not in df.columns:
            continue
            
        fig, ax1 = plt.subplots(figsize=(14, 7))
        
        # --- X-AXIS FIX: FORCE FULL 0-1 RANGE ---
        ax1.set_xlim(0, 1.0)
        ax1.set_xlabel('H_INERTIA (Full Sweep: Chaos → Stability)', fontsize=13)
        
        # --- PRIMARY AXIS (Metric 1) ---
        ax1.set_ylabel(m1_name, color=m1_color, fontsize=13, fontweight='bold')
        ax1.scatter(df_sweep['H_INERTIA'], df_sweep[m1_key], color=m1_color, alpha=0.07, s=10)
        
        # Trend Line: min_periods=1 prevents edge-clipping at 0.0 and 1.0
        m1_trend = df_sweep[m1_key].rolling(window=25, center=True, min_periods=1).mean()
        ax1.plot(df_sweep['H_INERTIA'], m1_trend, color=m1_color, linewidth=3, label=f'{m1_name} Trend')
        
        ax1.tick_params(axis='y', labelcolor=m1_color)
        ax1.grid(True, linestyle='--', alpha=0.15)

        # --- SECONDARY AXIS (Metric 2) ---
        ax2 = ax1.twinx()
        ax2.set_ylabel(m2_name, color=m2_color, fontsize=13, fontweight='bold')
        ax2.scatter(df_sweep['H_INERTIA'], df_sweep[m2_key], color=m2_color, alpha=0.07, s=10)
        
        m2_trend = df_sweep[m2_key].rolling(window=25, center=True, min_periods=1).mean()
        ax2.plot(df_sweep['H_INERTIA'], m2_trend, color=m2_color, linewidth=3, linestyle='--', label=f'{m2_name} Trend')
        
        ax2.tick_params(axis='y', labelcolor=m2_color)

        # --- GLOBAL OPTIMA REFERENCE ---
        # Find peak based on accuracy trend across the whole file
        acc_rolling = df_sweep['acc'].rolling(window=25, center=True, min_periods=1).mean()
        acc_peak_h = df_sweep.loc[acc_rolling.idxmax(), 'H_INERTIA']
        ax1.axvline(x=acc_peak_h, color='#e74c3c', linestyle=':', alpha=0.8, 
                    label=f'Peak Accuracy (H={acc_peak_h:.3f})')

        plt.title(f'Bifurcation: {m1_name} vs. {m2_name}\n[Full Parameter Space Sweep]', fontsize=16, fontweight='bold', pad=20)
        
        # Legend
        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', frameon=True)

        fig.tight_layout()
        
        save_name = f"FULL_RANGE_{m1_key.upper()}_vs_{m2_key.upper()}.png"
        plt.savefig(os.path.join(BIFURCATION_DIR, save_name), dpi=300, bbox_inches='tight')
        plt.close()

if __name__ == "__main__":
    generate_all_bifurcation_pairs(df)
    print(f"Plots saved to {BIFURCATION_DIR}")

In [ ]:
import os
import seaborn as sns
import matplotlib.pyplot as plt

# --- 1. SETTINGS ---

DENSITY_DIR = os.path.join(BASE_SAVE_PATH, "phase_density_maps")
os.makedirs(DENSITY_DIR, exist_ok=True)

# Define the metrics to map (Key, Display Name)
METRIC_LIST = [
    ("acc", "Accuracy"),
    ("rank", "Effective Rank"),
    ("sync", "Global Synchrony"),
    ("intf", "Interference"),
    ("entr", "Entropy"),
    ("acorr", "Auto-Correlation")
]

def generate_all_density_maps(df):
    print(f"Generating Phase Transition Maps for {len(METRIC_LIST)} metrics...")
    
    # Pre-calculate the global peak accuracy for a consistent reference line
    acc_peak_h = df.loc[df['acc'].rolling(window=20, min_periods=1, center=True).mean().idxmax(), 'H_INERTIA']

    for m_key, m_name in METRIC_LIST:
        if m_key not in df.columns:
            continue
            
        print(f" -> Mapping Density: {m_name}")
        plt.figure(figsize=(12, 8))
        
        # 1. KDE DENSITY (The "Topography")
        # 'rocket' provides a great 'heat' feel for scientific papers
        sns.kdeplot(
            data=df, 
            x="H_INERTIA", 
            y=m_key, 
            cmap="rocket", 
            fill=True, 
            thresh=0, 
            levels=35,
            alpha=0.9
        )
        
        # 2. LIGHT SCATTER OVERLAY
        plt.scatter(df['H_INERTIA'], df[m_key], color='white', s=1.5, alpha=0.1)

        # 3. AXIS LIMITS & REFERENCE
        #plt.xlim(0, 1.0)
        # Dynamic Y-limit with a 10% buffer
        #y_min, y_max = df[m_key].min(), df[m_key].max()
        #plt.ylim(y_min - abs(y_min*0.1), y_max + abs(y_max*0.1))
        
        # Draw the Critical Stability Line (Based on Accuracy Peak)
        #plt.axvline(x=acc_peak_h, color='#3ce73c', linestyle='--', linewidth=1.5, alpha=0.5)
        
        # 4. FORMATTING
        plt.title(f"Phase Map: {m_name} Density vs. H_INERTIA", fontsize=16, fontweight='bold', pad=20)
        plt.xlabel("H_INERTIA (Master Stability Parameter)", fontsize=13)
        plt.ylabel(f"{m_name} Value", fontsize=13)
        plt.grid(True, linestyle=':', alpha=0.15)

        # 5. SAVE
        save_name = f"DENSITY_MAP_{m_key.upper()}.png"
        plt.savefig(os.path.join(DENSITY_DIR, save_name), dpi=300, bbox_inches='tight')
        plt.close()

if __name__ == "__main__":
    generate_all_density_maps(df)
    print(f"\nSUCCESS: All density maps are ready in {DENSITY_DIR}")

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd
from itertools import combinations

# --- 1. SETTINGS ---
# Using the mapping provided for folder naming and variable selection
SWEEP_VAR = 'BASE_STRENGTH' 
VAR_LABEL = 'Base Strength ($\epsilon$)'
BIFURCATION_DIR = os.path.join(BASE_SAVE_PATH, "bifurcation_base_strength_sweep")
os.makedirs(BIFURCATION_DIR, exist_ok=True)

METRICS = [
    ('acc', 'Accuracy', '#2ecc71'),
    ('rank', 'Effective Rank', '#3498db'),
    ('sync', 'Synchrony', '#9b59b6'),
    ('intf', 'Interference', '#e67e22'),
    ('entr', 'Entropy', '#f1c40f'),
    ('acorr', 'Auto-Correlation', '#1abc9c')
]

def generate_strength_bifurcation_pairs(df):
    if SWEEP_VAR not in df.columns:
        print(f"Error: {SWEEP_VAR} not found in DataFrame.")
        return

    print(f"Starting Bifurcation Analysis for {VAR_LABEL}...")
    
    # Sort by the strength parameter
    df_sweep = df.sort_values(SWEEP_VAR).copy()
    
    metric_pairs = list(combinations(METRICS, 2))
    
    for (m1_key, m1_name, m1_color), (m2_key, m2_name, m2_color) in metric_pairs:
        if m1_key not in df.columns or m2_key not in df.columns:
            continue
            
        fig, ax1 = plt.subplots(figsize=(14, 7))
        
        # --- X-AXIS: DYNAMIC RANGE FOR STRENGTH ---
        ax1.set_xlabel(f'{VAR_LABEL} (Interaction Intensity)', fontsize=13)
        
        # --- PRIMARY AXIS ---
        ax1.set_ylabel(m1_name, color=m1_color, fontsize=13, fontweight='bold')
        ax1.scatter(df_sweep[SWEEP_VAR], df_sweep[m1_key], color=m1_color, alpha=0.07, s=10)
        
        m1_trend = df_sweep[m1_key].rolling(window=25, center=True, min_periods=1).mean()
        ax1.plot(df_sweep[SWEEP_VAR], m1_trend, color=m1_color, linewidth=3, label=f'{m1_name} Trend')
        
        ax1.tick_params(axis='y', labelcolor=m1_color)
        ax1.grid(True, linestyle='--', alpha=0.15)

        # --- SECONDARY AXIS ---
        ax2 = ax1.twinx()
        ax2.set_ylabel(m2_name, color=m2_color, fontsize=13, fontweight='bold')
        ax2.scatter(df_sweep[SWEEP_VAR], df_sweep[m2_key], color=m2_color, alpha=0.07, s=10)
        
        m2_trend = df_sweep[m2_key].rolling(window=25, center=True, min_periods=1).mean()
        ax2.plot(df_sweep[SWEEP_VAR], m2_trend, color=m2_color, linewidth=3, linestyle='--', label=f'{m2_name} Trend')
        
        ax2.tick_params(axis='y', labelcolor=m2_color)

        # --- GLOBAL OPTIMA REFERENCE (Based on Accuracy) ---
        acc_rolling = df_sweep['acc'].rolling(window=25, center=True, min_periods=1).mean()
        if not acc_rolling.isna().all():
            opt_val = df_sweep.loc[acc_rolling.idxmax(), SWEEP_VAR]
            ax1.axvline(x=opt_val, color='#e74c3c', linestyle=':', alpha=0.8, 
                        label=f'Optimal Strength ({opt_val:.3f})')

        plt.title(f'Bifurcation: {m1_name} vs. {m2_name}\n[Sweep: {VAR_LABEL}]', fontsize=16, fontweight='bold', pad=20)
        
        # Merge legends
        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', frameon=True)

        fig.tight_layout()
        
        save_name = f"STRENGTH_SWEEP_{m1_key.upper()}_vs_{m2_key.upper()}.png"
        plt.savefig(os.path.join(BIFURCATION_DIR, save_name), dpi=300, bbox_inches='tight')
        plt.close()

if __name__ == "__main__":
    generate_strength_bifurcation_pairs(df)
    print(f"Strength sweep plots saved to {BIFURCATION_DIR}")